## Notes on CAPSTONE Project

In [522]:
import pandas as pd
import numpy as np
import json
import math
import time
import requests
from math import radians, sin, cos, sqrt, atan2
from scipy import stats

In [520]:
base = '/Users/isabellawoods/Documents/northwestern 25-26/CAPSTONE'
traffic = pd.read_csv(f'{base}/DATA/Traffic.csv')
visit   = pd.read_csv(f'{base}/DATA/Visitation.csv', skiprows = 2)
species = pd.read_csv(f'{base}/NPSpecies.csv')

/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_80408/869447368.py:4: DtypeWarning: Columns (22,27) have mixed types. Specify dtype option on import or set low_memory=False.
  species = pd.read_csv(f'{base}/NPSpecies.csv')


In [536]:

# first clean traffic count — remove commas and convert to numeric
traffic['TrafficCount_clean'] = pd.to_numeric(
    traffic['TrafficCount'].astype(str).str.replace(',', ''), errors='coerce'
)
traffic['TrafficCountTotal_clean'] = pd.to_numeric(
    traffic['TrafficCountTotal'].astype(str).str.replace(',', ''), errors='coerce'
)

# aggregate to annual total per park
annual_traffic = (traffic
    .groupby(['UnitCode', 'ParkName', 'Year'])['TrafficCount_clean']
    .sum()
    .reset_index()
    .rename(columns={'TrafficCount_clean': 'annual_traffic'})
)

def traffic_features(group):
    group = group[['Year', 'annual_traffic']].sort_values('Year')  # only keep needed cols
    years = group['Year'].values
    counts = group['annual_traffic'].values
    
    if len(group) < 3:
        return pd.Series({
            'traffic_slope':           np.nan,
            'traffic_acceleration':    np.nan,
            'avg_annual_traffic':      counts.mean(),
            'max_annual_traffic':      counts.max(),
            'recent_traffic':          counts[-1],
            'traffic_cv':              counts.std() / counts.mean() if counts.mean() > 0 else np.nan,
            'traffic_covid_impact':    np.nan,
        })
    
    slope, _, _, _, _ = stats.linregress(years, counts)
    acceleration = counts[-3:].mean() - counts[:3].mean()
    
    pre_covid  = group[group['Year'] == 2019]['annual_traffic'].values
    post_covid = group[group['Year'] == 2020]['annual_traffic'].values
    covid_impact = ((post_covid[0] - pre_covid[0]) / pre_covid[0] * 100 
                    if len(pre_covid) > 0 and len(post_covid) > 0 and pre_covid[0] > 0 
                    else np.nan)
    
    return pd.Series({
        'traffic_slope':        slope,
        'traffic_acceleration': acceleration,
        'avg_annual_traffic':   counts.mean(),
        'max_annual_traffic':   counts.max(),
        'recent_traffic':       counts[-1],
        'traffic_cv':           counts.std() / counts.mean() if counts.mean() > 0 else np.nan,
        'traffic_covid_impact': covid_impact,
    })

traffic_features_df = (annual_traffic
    .groupby(['UnitCode', 'ParkName'])
    .apply(traffic_features)
    .reset_index()
    .drop(columns='level_2', errors='ignore')
)

seasonality = (traffic
    .groupby(['UnitCode', 'Year', 'Month'])['TrafficCount_clean']
    .sum()
    .reset_index()
    .groupby(['UnitCode', 'Year'])
    .apply(lambda g: g['TrafficCount_clean'].max() / g['TrafficCount_clean'].min() 
           if g['TrafficCount_clean'].min() > 0 else np.nan)
    .reset_index()
    .groupby('UnitCode')[0]
    .mean()
    .reset_index()
    .rename(columns={0: 'traffic_seasonality'})
)

traffic_features_df = traffic_features_df.merge(seasonality, on='UnitCode', how='left')


traffic_features_df.head()

,UnitCode,ParkName,traffic_slope,traffic_acceleration,avg_annual_traffic,max_annual_traffic,recent_traffic,traffic_cv,traffic_covid_impact,traffic_seasonality
0,ABLI,Abraham Lincoln Birthplace NHP,-396.762366,-9843.000000,74210.484848,109535.0,63305.0,0.189114,-2.898903,6.787155
1,ACAD,Acadia NP,810.728057,101119.000000,445388.583333,597569.0,594835.0,0.198250,-17.227075,17.937362
2,ALPO,Allegheny Portage Railroad NHS,1056.352607,20975.333333,67726.606061,92475.0,76883.0,0.206278,-25.589087,3.356113
3,AMCH,Amache NHS,NaN,NaN,3032.500000,3747.0,3747.0,0.235614,NaN,4.166219
4,AMIS,Amistad NRA,-17487.412605,-579963.666667,480944.371429,889687.0,216355.0,0.415894,15.657959,2.881021


In [544]:
visit = visit.drop(columns = ['TotalRecreationVisitors', 'TotalNonRecreationVisitors',
       'TotalRecreationHours', 'TotalNonRecreationHours',
       'TotalConcessionerLodging', 'TotalConcessionerCamping',
       'TotalTentCampers', 'TotalRVCampers', 'TotalBackcountry',
       'TotalNonRecreationOvernightStays', 'TotalMiscellaneousOvernightStays'])

In [546]:
# the downloaded csv contained two tables appended to each other
# row of column headers in the middle of the df
visit.index[visit['Year'] == 'Reporting']
# park visits is visitors by park
parkvisits = visit.iloc[:24372, :]

# total visitation numbers across all parks by year
totalvisits = visit.iloc[24372:, :]
# set columns as row 24372
totalvisits.columns = totalvisits.iloc[0]
totalvisits = totalvisits.iloc[1:]

In [500]:
# select all unique park names
pv_parks = parkvisits.ParkName.unique()
traffic_name = traffic.ParkName.unique()
traffic_code = traffic.UnitCode.unique()
species_parks = species["Park Code"].unique()

# select parks only present in all 3
traffic_species = set(traffic_code) & set(species_parks)
traffic_visits = set(traffic_name) & set(pv_parks)

print(f"Parks in traffic+species: {len(traffic_species)}")

code_to_name = dict(zip(traffic.UnitCode, traffic.ParkName))
name_to_code = dict(zip(traffic.ParkName, traffic.UnitCode))  # fixed: was name_to_comde

traffic_species_names = {code_to_name[code] for code in traffic_species if code in code_to_name}
common = set(traffic_species_names) & set(traffic_visits)

common_codes = {name_to_code[name] for name in common if name in name_to_code}  # fixed: namme -> name, name_to_comde -> name_to_code

print(f"Parks in all three datasets: {len(common)}")
print(f"Common codes: {len(common_codes)}")

Parks in traffic+species: 210
Parks in all three datasets: 210
Common codes: 210


### Visitation Data

In [575]:
parkvisits['Code'] = parkvisits['ParkName'].map(name_to_code)
def to_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',', ''), errors='coerce').fillna(0)

def covid_pct(group, col):
    pre  = to_numeric(group[group['Year'] == 2019][col]).values
    post = to_numeric(group[group['Year'] == 2020][col]).values
    if len(pre) > 0 and len(post) > 0 and pre[0] > 0:
        return (post[0] - pre[0]) / pre[0] * 100
    return np.nan

def get_slope(group, col):
    years  = group['Year'].astype(float).values  # force float
    values = to_numeric(group[col]).values.astype(float)  # force float
    mask   = ~np.isnan(values) & (values > 0)
    if mask.sum() < 3:
        return np.nan
    return lr(years[mask], values[mask])[0]

most_recent_year = parkvisits['Year'].max()
    
def get_recent(group, col):
    row = group[group['Year'] == most_recent_year][col]
    if len(row) > 0:
        return to_numeric(row).values[0]
    return np.nan

def visit_features(g):
    return pd.Series({
        f'recreation_visitors_{most_recent_year}':  get_recent(g, 'RecreationVisitors'),
        f'nonrec_visitors_{most_recent_year}':      get_recent(g, 'NonRecreationVisitors'),
        f'recreation_hours_{most_recent_year}':     get_recent(g, 'RecreationHours'),
        f'tent_campers_{most_recent_year}':         get_recent(g, 'TentCampers'),
        f'rv_campers_{most_recent_year}':           get_recent(g, 'RVCampers'),
        f'backcountry_{most_recent_year}':          get_recent(g, 'Backcountry'),
        f'lodging_{most_recent_year}':              get_recent(g, 'ConcessionerLodging'),

        # --- long run averages ---
        'avg_recreation_visitors':      to_numeric(g['RecreationVisitors']).mean(),
        'max_recreation_visitors':      to_numeric(g['RecreationVisitors']).max(),
        'avg_tent_campers':             to_numeric(g['TentCampers']).mean(),
        'avg_rv_campers':               to_numeric(g['RVCampers']).mean(),
        'avg_backcountry':              to_numeric(g['Backcountry']).mean(),
        'avg_lodging':                  to_numeric(g['ConcessionerLodging']).mean(),

        # --- slopes (change over time) ---
        'visit_slope':                  get_slope(g, 'RecreationVisitors'),
        'tent_campers_slope':           get_slope(g, 'TentCampers'),
        'rv_campers_slope':             get_slope(g, 'RVCampers'),
        'backcountry_slope':            get_slope(g, 'Backcountry'),
        'lodging_slope':                get_slope(g, 'ConcessionerLodging'),
        'hours_slope':                  get_slope(g, 'RecreationHours'),

        # --- derived ratios ---
        'hours_per_visitor':            (to_numeric(g['RecreationHours']).sum() /
                                         to_numeric(g['RecreationVisitors']).sum()),
        'overnight_ratio':              ((to_numeric(g['TentCampers']) +
                                          to_numeric(g['RVCampers']) +
                                          to_numeric(g['ConcessionerLodging'])).mean() /
                                          to_numeric(g['RecreationVisitors']).mean()),
        'nonrec_ratio':                 (to_numeric(g['NonRecreationVisitors']).mean() /
                                         to_numeric(g['RecreationVisitors']).mean()),

        # --- covid impact ---
        'visit_covid_impact':           covid_pct(g, 'RecreationVisitors'),
        'backcountry_covid_impact':     covid_pct(g, 'Backcountry'),
    })

visit_features_df = (parkvisits
    .groupby('Code')
    .apply(visit_features)
    .reset_index()
)

visit_features_df.head()

/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_80408/1221308032.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parkvisits['Code'] = parkvisits['ParkName'].map(name_to_code)


,Code,recreation_visitors_2025,nonrec_visitors_2025,recreation_hours_2025,tent_campers_2025,rv_campers_2025,backcountry_2025,lodging_2025,avg_recreation_visitors,max_recreation_visitors,...,tent_campers_slope,rv_campers_slope,backcountry_slope,lodging_slope,hours_slope,hours_per_visitor,overnight_ratio,nonrec_ratio,visit_covid_impact,backcountry_covid_impact
0,ABLI,212899.0,0.0,212899.0,0.0,0.0,0.0,0.0,2.495205e+05,557439.0,...,NaN,NaN,NaN,NaN,-1616.879163,0.500658,0.000000,0.000000,NaN,NaN
1,ACAD,4079318.0,47100.0,27250620.0,128931.0,52795.0,1288.0,0.0,1.837622e+06,5440952.0,...,-744.279978,-352.363302,16.939160,NaN,-40184.285153,4.356309,0.040141,0.036194,NaN,NaN
2,ALPO,162905.0,180.0,266932.0,0.0,0.0,0.0,0.0,1.148785e+05,201837.0,...,NaN,NaN,NaN,NaN,4843.197387,1.229820,0.000005,0.002730,NaN,NaN
3,AMCH,8113.0,300.0,10030.0,0.0,0.0,0.0,0.0,6.442000e+03,8113.0,...,NaN,NaN,NaN,NaN,NaN,1.148789,0.000000,0.049364,NaN,NaN
4,AMIS,734293.0,90.0,5453812.0,0.0,7244.0,0.0,0.0,1.208808e+06,2573966.0,...,-683.877966,-1230.073312,-721.668783,NaN,-21470.663969,6.512024,0.021915,0.001819,NaN,NaN


In [577]:
visit_traffic_ft = pd.merge(visit_features_df, traffic_features_df, how = "inner", left_on = "Code", right_on = "UnitCode")
visit_traffic_ft

,Code,recreation_visitors_2025,nonrec_visitors_2025,recreation_hours_2025,tent_campers_2025,rv_campers_2025,backcountry_2025,lodging_2025,avg_recreation_visitors,max_recreation_visitors,...,UnitCode,ParkName,traffic_slope,traffic_acceleration,avg_annual_traffic,max_annual_traffic,recent_traffic,traffic_cv,traffic_covid_impact,traffic_seasonality
0,ABLI,212899.0,0.0,212899.0,0.0,0.0,0.0,0.0,2.495205e+05,557439.0,...,ABLI,Abraham Lincoln Birthplace NHP,-396.762366,-9.843000e+03,7.421048e+04,109535.0,63305.0,0.189114,-2.898903,6.787155
1,ACAD,4079318.0,47100.0,27250620.0,128931.0,52795.0,1288.0,0.0,1.837622e+06,5440952.0,...,ACAD,Acadia NP,810.728057,1.011190e+05,4.453886e+05,597569.0,594835.0,0.198250,-17.227075,17.937362
2,ALPO,162905.0,180.0,266932.0,0.0,0.0,0.0,0.0,1.148785e+05,201837.0,...,ALPO,Allegheny Portage Railroad NHS,1056.352607,2.097533e+04,6.772661e+04,92475.0,76883.0,0.206278,-25.589087,3.356113
3,AMCH,8113.0,300.0,10030.0,0.0,0.0,0.0,0.0,6.442000e+03,8113.0,...,AMCH,Amache NHS,NaN,NaN,3.032500e+03,3747.0,3747.0,0.235614,NaN,4.166219
4,AMIS,734293.0,90.0,5453812.0,0.0,7244.0,0.0,0.0,1.208808e+06,2573966.0,...,AMIS,Amistad NRA,-17487.412605,-5.799637e+05,4.809444e+05,889687.0,216355.0,0.415894,15.657959,2.881021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,WICR,370558.0,0.0,741117.0,0.0,0.0,0.0,0.0,1.644620e+05,370558.0,...,WICR,Wilson's Creek NB,-235.164986,3.523733e+04,7.520780e+04,106015.0,102665.0,0.248981,20.719504,5.354439
223,WRBR,341576.0,2190.0,341576.0,0.0,0.0,0.0,0.0,3.331937e+05,712181.0,...,WRBR,Wright Brothers NMEM,-141.474265,-1.264300e+04,1.342931e+05,181935.0,106932.0,0.124366,-26.466467,12.728446
224,YELL,4762988.0,1311542.0,86891452.0,53210.0,37261.0,36895.0,692725.0,1.748012e+06,4860242.0,...,YELL,Yellowstone NP,46382.271429,2.022890e+06,2.144692e+06,3161171.0,3088488.0,0.261096,-2.119504,11.404340
225,YOSE,4278413.0,171513.0,64185755.0,439233.0,264908.0,139743.0,666379.0,1.872454e+06,5028868.0,...,YOSE,Yosemite NP,NaN,NaN,1.873423e+06,1873423.0,1873423.0,0.000000,NaN,5.336688


In [9]:
parks = [
    'Abraham Lincoln Birthplace National Historical Park, US, KY',
    'Acadia National Park, US, ME',
    'Allegheny Portage Railroad National Historic Site, US, PA',
    'Amistad National Recreation Area, US, TX',
    'Apostle Islands National Lakeshore, US, WI',
    'Appomattox Court House National Historical Park, US, VA',
    'Arches National Park, US, UT',
    'Arkansas Post National Memorial, US, AR',
    'Assateague Island National Seashore, US, MD',
    'Badlands National Park, US, SD',
    'Bandelier National Monument, US, NM',
    'Big Bend National Park, US, TX',
    'Big Cypress National Preserve, US, FL',
    'Big Hole National Battlefield, US, MT',
    'Big South Fork National River and Recreation Area, US, TN',
    'Big Thicket National Preserve, US, TX',
    'Bighorn Canyon National Recreation Area, US, MT',
    'Biscayne National Park, US, FL',
    'Black Canyon of the Gunnison National Park, US, CO',
    'Blue Ridge Parkway, US, VA',
    'Booker T. Washington National Monument, US, VA',
    'Bryce Canyon National Park, US, UT',
    'Buffalo National River, US, AR',
    'Cabrillo National Monument, US, CA',
    'Camp Nelson National Monument, US, KY',
    'Canaveral National Seashore, US, FL',
    'Cane River Creole National Historical Park, US, LA',
    'Canyon de Chelly National Monument, US, AZ',
    'Canyonlands National Park, US, UT',
    'Cape Cod National Seashore, US, MA',
    'Cape Hatteras National Seashore, US, NC',
    'Cape Lookout National Seashore, US, NC',
    'Capitol Reef National Park, US, UT',
    'Capulin Volcano National Monument, US, NM',
    'Carlsbad Caverns National Park, US, NM',
    'Catoctin Mountain Park, US, MD',
    'Cedar Breaks National Monument, US, UT',
    'Chaco Culture National Historical Park, US, NM',
    'Charles Pinckney National Historic Site, US, SC',
    'Chattahoochee River National Recreation Area, US, GA',
    'Chesapeake and Ohio Canal National Historical Park, US, MD',
    'Chickamauga and Chattanooga National Military Park, US, GA',
    'Chickasaw National Recreation Area, US, OK',
    'City of Rocks National Reserve, US, ID',
    'Colonial National Historical Park, US, VA',
    'Colorado National Monument, US, CO',
    'Congaree National Park, US, SC',
    'Coronado National Memorial, US, AZ',
    'Cowpens National Battlefield, US, SC',
    'Crater Lake National Park, US, OR',
    'Craters of the Moon National Monument and Preserve, US, ID',
    'Cumberland Gap National Historical Park, US, KY',
    'Curecanti National Recreation Area, US, CO',
    'Cuyahoga Valley National Park, US, OH',
    'De Soto National Memorial, US, FL',
    'Death Valley National Park, US, CA',
    'Delaware Water Gap National Recreation Area, US, PA',
    'Denali National Park and Preserve, US, AK',
    'Devils Postpile National Monument, US, CA',
    'Devils Tower National Monument, US, WY',
    'Dinosaur National Monument, US, CO',
    'Eisenhower National Historic Site, US, PA',
    'El Malpais National Monument, US, NM',
    'El Morro National Monument, US, NM',
    'Everglades National Park, US, FL',
    'Fire Island National Seashore, US, NY',
    'Florissant Fossil Beds National Monument, US, CO',
    'Fort Caroline National Memorial, US, FL',
    'Fort Donelson National Battlefield, US, TN',
    'Fort Frederica National Monument, US, GA',
    'Fort Laramie National Historic Site, US, WY',
    'Fort Larned National Historic Site, US, KS',
    'Fort Matanzas National Monument, US, FL',
    'Fort Necessity National Battlefield, US, PA',
    'Fort Point National Historic Site, US, CA',
    'Fort Pulaski National Monument, US, GA',
    'Fort Raleigh National Historic Site, US, NC',
    'Fort Vancouver National Historic Site, US, WA',
    'Fort Washington Park, US, MD',
    'Fossil Butte National Monument, US, WY',
    'Fredericksburg and Spotsylvania National Military Park, US, VA',
    'Gateway National Recreation Area, US, NY',
    'Gauley River National Recreation Area, US, WV',
    'George Washington Birthplace National Monument, US, VA',
    'George Washington Carver National Monument, US, MO',
    'George Washington Memorial Parkway, US, VA',
    'Gettysburg National Military Park, US, PA',
    'Glacier National Park, US, MT',
    'Glen Canyon National Recreation Area, US, UT',
    'Golden Gate National Recreation Area, US, CA',
    'Grand Canyon National Park, US, AZ',
    'Grand Portage National Monument, US, MN',
    'Grand Teton National Park, US, WY',
    'Great Basin National Park, US, NV',
    'Great Sand Dunes National Park and Preserve, US, CO',
    'Great Smoky Mountains National Park, US, TN',
    'Greenbelt Park, US, MD',
    'Guadalupe Mountains National Park, US, TX',
    'Guilford Courthouse National Military Park, US, NC',
    'Gulf Islands National Seashore, US, FL',
    'Haleakala National Park, US, HI',
    'Harpers Ferry National Historical Park, US, WV',
    'Hawaii Volcanoes National Park, US, HI',
    'Home of Franklin D. Roosevelt National Historic Site, US, NY',
    'Homestead National Historical Park, US, NE',
    'Hopewell Culture National Historical Park, US, OH',
    'Hopewell Furnace National Historic Site, US, PA',
    'Horseshoe Bend National Military Park, US, AL',
    'Hot Springs National Park, US, AR',
    'Hubbell Trading Post National Historic Site, US, AZ',
    'Indiana Dunes National Park, US, IN',
    'Jean Lafitte National Historical Park and Preserve, US, LA',
    'Jewel Cave National Monument, US, SD',
    'John Day Fossil Beds National Monument, US, OR',
    'Johnstown Flood National Memorial, US, PA',
    'Joshua Tree National Park, US, CA',
    'Katahdin Woods and Waters National Monument, US, ME',
    'Katmai National Park and Preserve, US, AK',
    'Kenai Fjords National Park, US, AK',
    'Kennesaw Mountain National Battlefield Park, US, GA',
    'Kings Mountain National Military Park, US, SC',
    'LBJ Memorial Grove on the Potomac, US, VA',
    'Lake Mead National Recreation Area, US, NV',
    'Lake Meredith National Recreation Area, US, TX',
    'Lake Roosevelt National Recreation Area, US, WA',
    'Lassen Volcanic National Park, US, CA',
    'Lava Beds National Monument, US, CA',
    'Lewis and Clark National Historical Park, US, OR',
    'Lincoln Boyhood National Memorial, US, IN',
    'Little Bighorn Battlefield National Monument, US, MT',
    'Little River Canyon National Preserve, US, AL',
    'Lyndon B. Johnson National Historical Park, US, TX',
    'Mammoth Cave National Park, US, KY',
    'Manassas National Battlefield Park, US, VA',
    'Manzanar National Historic Site, US, CA',
    'Martin Van Buren National Historic Site, US, NY',
    'Mesa Verde National Park, US, CO',
    'Minute Man National Historical Park, US, MA',
    'Missouri National Recreational River, US, NE',
    'Mojave National Preserve, US, CA',
    'Monocacy National Battlefield, US, MD',
    'Montezuma Castle National Monument, US, AZ',
    'Moores Creek National Battlefield, US, NC',
    'Morristown National Historical Park, US, NJ',
    'Mount Rainier National Park, US, WA',
    'Mount Rushmore National Memorial, US, SD',
    'Natchez Trace Parkway, US, MS',
    'Natural Bridges National Monument, US, UT',
    'Navajo National Monument, US, AZ',
    'New River Gorge National Park and Preserve, US, WV',
    'Nez Perce National Historical Park, US, ID',
    'Ninety Six National Historic Site, US, SC',
    'North Cascades National Park, US, WA',
    'Obed Wild and Scenic River, US, TN',
    'Ocmulgee Mounds National Historical Park, US, GA',
    'Olympic National Park, US, WA',
    'Oregon Caves National Monument and Preserve, US, OR',
    'Organ Pipe Cactus National Monument, US, AZ',
    'Ozark National Scenic Riverways, US, MO',
    'Padre Island National Seashore, US, TX',
    'Palo Alto Battlefield National Historical Park, US, TX',
    'Pea Ridge National Military Park, US, AR',
    'Petersburg National Battlefield, US, VA',
    'Petrified Forest National Park, US, AZ',
    'Petroglyph National Monument, US, NM',
    'Pictured Rocks National Lakeshore, US, MI',
    'Pinnacles National Park, US, CA',
    'Pipe Spring National Monument, US, AZ',
    'Piscataway Park, US, MD',
    'Point Reyes National Seashore, US, CA',
    'Prince William Forest Park, US, VA',
    "Pu'ukohola Heiau National Historic Site, US, HI",
    'Redwood National Park, US, CA',
    'Richmond National Battlefield Park, US, VA',
    'Rocky Mountain National Park, US, CO',
    'Russell Cave National Monument, US, AL',
    'Saguaro National Park, US, AZ',
    'Saint-Gaudens National Historical Park, US, NH',
    'San Antonio Missions National Historical Park, US, TX',
    'San Juan Island National Historical Park, US, WA',
    'Santa Monica Mountains National Recreation Area, US, CA',
    'Saratoga National Historical Park, US, NY',
    'Scotts Bluff National Monument, US, NE',
    'Shenandoah National Park, US, VA',
    'Shiloh National Military Park, US, TN',
    'Sleeping Bear Dunes National Lakeshore, US, MI',
    'Stones River National Battlefield, US, TN',
    'Sunset Crater Volcano National Monument, US, AZ',
    'Tallgrass Prairie National Preserve, US, KS',
    'Theodore Roosevelt National Park, US, ND',
    'Timucuan Ecological and Historic Preserve, US, FL',
    'Tonto National Monument, US, AZ',
    'Tuzigoot National Monument, US, AZ',
    'Upper Delaware Scenic and Recreational River, US, NY',
    'Valles Caldera National Preserve, US, NM',
    'Valley Forge National Historical Park, US, PA',
    'Vanderbilt Mansion National Historic Site, US, NY',
    'Vicksburg National Military Park, US, MS',
    'Walnut Canyon National Monument, US, AZ',
    'War in the Pacific National Historical Park, US, GU',
    'Washita Battlefield National Historic Site, US, OK',
    'Whiskeytown National Recreation Area, US, CA',
    'White Sands National Park, US, NM',
    'Whitman Mission National Historic Site, US, WA',
    "Wilson's Creek National Battlefield, US, MO",
    'Wind Cave National Park, US, SD',
    'Wright Brothers National Memorial, US, NC',
    'Yellowstone National Park, US, WY',
    'Yosemite National Park, US, CA',
    'Zion National Park, US, UT',
]

import requests, zipfile, io
r = requests.get('http://www.inaturalist.org/places/inaturalist-places.csv.zip', stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall('/Users/isabellawoods/Documents/northwestern 25-26/CAPSTONE/DATA')

## Scrape iNaturalist + SDI

In [11]:
places = pd.read_csv('/Users/isabellawoods/Documents/northwestern 25-26/CAPSTONE/DATA/inaturalist-places.csv')

my_places = []
not_found = []

for park in parks:
    matches = places[places['display_name'].str.contains(park, case=False, na=False)]
    my_places.append(matches)
    if len(matches) == 0:
        not_found.append(park)

poi = pd.concat(my_places) #parks of interest
len(poi)

170

In [12]:
import math
import time
import requests
import pandas as pd

def fetch_with_retry(url, max_retries=5):
    """Fetch a URL with exponential backoff on failure."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                return response.json()
            else:
                print(f'  Status {response.status_code}, retrying...')
        except (requests.exceptions.ConnectionError, 
                requests.exceptions.Timeout) as e:
            wait = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
            print(f'  Connection error: {e}. Waiting {wait}s...')
            time.sleep(wait)
    print(f'  Failed after {max_retries} attempts: {url}')
    return None

start_time = time.time()
base = 'https://api.inaturalist.org/v1/observations?place_id='
park_species = []

cols = {
    'id':                          'obs_id',
    'uuid':                        'uuid',
    'observed_on':                 'observed_on',
    'quality_grade':               'quality_grade',
    'place_guess':                 'place_guess',
    'location':                    'location',
    'captive':                     'captive',
    'taxon.name':                  'ScientificName',
    'taxon.preferred_common_name': 'CommonName',
    'taxon.rank':                  'TaxonRank',
    'taxon.iconic_taxon_name':     'TaxonGroup',
    'taxon.native':                'native',
    'taxon.introduced':            'introduced',
    'user.login':                  'user',
}

for i in range(len(poi)):
    place_id = poi.iloc[i]['id']
    place_name = poi.iloc[i]['display_name']

    all_pages = []
    last_id = 0

    while True:
        url = f'{base}{place_id}&per_page=200&order=asc&order_by=id&id_above={last_id}'
        data = fetch_with_retry(url)

        if data is None or 'results' not in data or len(data['results']) == 0:
            break

        results = data['results']

        df = pd.json_normalize(results)
        existing_cols = {k: v for k, v in cols.items() if k in df.columns}
        df = df[existing_cols.keys()].rename(columns=existing_cols)

        if 'location' in df.columns:
            df[['latitude', 'longitude']] = df['location'].str.split(',', expand=True).astype(float)
            df = df.drop(columns='location')

        df['park_name'] = place_name
        all_pages.append(df)

        last_id = results[-1]['id']
        time.sleep(1.5)  # slightly longer sleep to be safer

    park_df = pd.concat(all_pages, ignore_index=True) if all_pages else pd.DataFrame()
    park_species.append(park_df)

    elapsed = (time.time() - start_time) / 60
    print(f'[{i+1}/{len(poi)}] {place_name} — {len(park_df)} records | Elapsed: {elapsed:.1f} min')

iNaturalist = pd.concat(park_species, ignore_index=True)
print(f'Total records: {len(iNaturalist)}')

[1/170] Abraham Lincoln Birthplace National Historical Park, US, KY — 823 records | Elapsed: 0.3 min


KeyboardInterrupt: 

In [295]:
iNaturalist.to_csv('iNaturalist.csv', index=False)

In [15]:
# some of the parks were not scraped. I'm going to retry with the failed parks
failed_place_ids = [
    4504,   # Death Valley (partial - failed mid-fetch)
    5232,   # Devils Postpile
    95159,  # Devils Tower
    95164,  # Dinosaur
    95169,  # Eisenhower
    95173,  # El Malpais
    95174,  # El Morro
    53957,  # Everglades
    95160,  # Fire Island
    95152,  # Florissant Fossil Beds
    95163,  # Fort Caroline
    95184,  # Fort Frederica
    95185,  # Fort Laramie
    95187,  # Fort Matanzas
    95189,  # Fort Necessity
    95170,  # Fort Point
    95171,  # Fort Pulaski
    95172,  # Fort Raleigh
    95193,  # Fort Vancouver
    95183,  # Fossil Butte
    95180,  # Fredericksburg
    95202,  # Gauley River
    72881,  # George Washington Birthplace
    95224,  # George Washington Carver
    95208,  # Gettysburg
    72841,  # Glacier
    9912,   # Golden Gate
    69216,  # Grand Canyon
    95216,  # Grand Portage
    69099,  # Grand Teton
    69699,  # Great Basin
    69313,  # Guadalupe Mountains
    95214,  # Guilford Courthouse
    95223,  # Gulf Islands
    56788,  # Haleakala
    7222,   # Hawaii Volcanoes
    95205,  # Homestead
    95221,  # Hopewell Culture
    95231,  # Hopewell Furnace
    95227,  # Horseshoe Bend
    56706,  # Hot Springs
    95237,  # Hubbell Trading Post
    54005,  # Jean Lafitte
    95228,  # Jewel Cave
    95248,  # John Day Fossil Beds
    95254,  # Johnstown Flood
    3680,   # Joshua Tree
    95258,  # Kenai Fjords
    95249,  # Kennesaw Mountain
    95271,  # Lake Meredith
    95272,  # Lake Roosevelt
    4509,   # Lassen Volcanic
    4512,   # Lava Beds
    95274,  # Lewis and Clark
    95222,  # Lincoln Boyhood
    95261,  # Little Bighorn
    95234,  # Little River Canyon
    95275,  # Lyndon B. Johnson
    51988,  # Mammoth Cave
    72649,  # Mammoth Cave (duplicate)
    97441,  # Manassas
    4513,   # Manzanar
    95270,  # Martin Van Buren
    69108,  # Mesa Verde
    95279,  # Minute Man
    3958,   # Mojave
    97466,  # Monocacy
    95155,  # Montezuma Castle
    95158,  # Moores Creek
    95280,  # Morristown
    8838,   # Mount Rainier
    95281,  # Mount Rushmore
    95285,  # Natural Bridges
    95286,  # Navajo
    95209,  # New River Gorge
    95197,  # Nez Perce
    69097,  # North Cascades
    95295,  # Organ Pipe Cactus
    95300,  # Padre Island
    95297,  # Palo Alto Battlefield
    95302,  # Pea Ridge
    95306,  # Petersburg
    57573,  # Petrified Forest
    95307,  # Petroglyph
    95250,  # Pictured Rocks
    5737,   # Pinnacles
    95251,  # Pipe Spring
    97465,  # Piscataway
    5781,   # Point Reyes
    97468,  # Prince William Forest
    72770,  # Pu'ukohola Heiau
    6021,   # Redwood
    95267,  # Richmond Battlefield
    49676,  # Rocky Mountain
    95301,  # Russell Cave
    65739,  # Saguaro
    95303,  # San Antonio Missions
    95312,  # San Juan Island
    3730,   # Santa Monica Mountains
    95320,  # Saratoga
    95329,  # Scotts Bluff
    9012,   # Shenandoah
    95326,  # Sleeping Bear Dunes
    95329,  # Sunset Crater
    72793,  # Theodore Roosevelt
    95147,  # Timucuan
    95162,  # Tonto
    95181,  # Tuzigoot
    95211,  # Upper Delaware
    63030,  # Valley Forge
    95335,  # Vanderbilt Mansion
    95341,  # Walnut Canyon
    95346,  # Washita Battlefield
    62621,  # White Sands
    95343,  # Whitman Mission
    95344,  # Wilson's Creek
    72794,  # Wind Cave
    95346,  # Wright Brothers
    10211,  # Yellowstone
    68542,  # Yosemite
    50634,  # Zion
]

In [25]:
poi_failed = poi[poi['id'].isin(failed_place_ids)]
print(f"Parks to retry: {len(poi_failed)}")
poi_failed

Parks to retry: 119


,id,name,display_name,code,latitude,longitude,swlat,swlng,nelat,nelng,place_type,bbox_area,created_at,updated_at,ancestry,slug,source_id,admin_level,uuid,woeid
7704,4504,Death Valley National Park,"Death Valley National Park, US, CA",NaN,36.483899,-117.132594,35.633538,-118.001032,37.349445,-116.276524,100.0,2.959096,2009-06-30 05:31:52,2018-09-06 14:33:19.08021,97394/1/14,death-valley-national-park,10358.0,100.0,fa8ecb3d-c714-459b-9d22-43b8b9622e7b,NaN
7702,5232,Devils Postpile National Monument,"Devils Postpile National Monument, US, CA",NaN,37.615374,-119.087686,37.597256,-119.092242,37.633492,-119.083131,100.0,0.000330,2009-06-30 05:36:55,2018-09-06 14:33:18.978653,97394/1/14,devils-postpile-national-monument,10358.0,100.0,c933f99d-010f-4fb9-bdfb-6f8a8d8e62cd,NaN
71145,95159,Devils Tower National Monument,"Devils Tower National Monument, US, WY",NaN,44.590649,-104.715642,44.578960,-104.730327,44.601201,-104.700022,100.0,0.000674,2016-01-26 07:33:59.935243,2018-09-06 14:33:19.070401,97394/1/15,devils-tower-national-monument,10358.0,100.0,0a7c83b0-1731-45c2-a428-e9f3682f7e08,NaN
71150,95164,Dinosaur National Monument,"Dinosaur National Monument, US, CO",NaN,40.507273,-108.933266,40.240890,-109.337661,40.746814,-108.341355,100.0,0.504055,2016-01-26 07:34:06.930846,2018-09-06 14:33:19.143975,97394/1/34,dinosaur-national-monument,10358.0,100.0,aa3d6fae-8667-4165-b650-ba139051a023,NaN
71155,95169,Eisenhower National Historic Site,"Eisenhower National Historic Site, US, PA",NaN,39.795768,-77.265088,39.782876,-77.279623,39.809957,-77.255988,100.0,0.000640,2016-01-26 07:34:12.313327,2018-09-06 14:33:19.21188,97394/1/42,eisenhower-national-historic-site,10358.0,100.0,81976931-d51d-403a-990d-9d91239dace0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59835,72794,Wind Cave National Park,"Wind Cave National Park, US, SD",NaN,43.580061,-103.439456,43.497197,-103.550635,43.640484,-103.336808,100.0,0.030639,2014-11-13 04:16:32.694881,2018-09-06 14:33:21.445178,97394/1/44,wind-cave,10358.0,100.0,88761e10-f81b-424b-837e-bf77c67d7d83,NaN
71302,95344,Wright Brothers National Memorial,"Wright Brothers National Memorial, US, NC",NaN,36.016377,-75.669966,36.008008,-75.678545,36.025136,-75.657707,100.0,0.000357,2016-01-26 07:38:45.859383,2018-09-06 14:33:21.585078,97394/1/30,wright-brothers-national-memorial,10358.0,100.0,1436fc8b-f787-4a28-a39a-a7f4f328cabe,NaN
20571,10211,Yellowstone National Park,"Yellowstone National Park, US, WY",NaN,44.596414,-110.547110,44.132584,-111.155986,45.108957,-109.824180,100.0,1.300339,2011-07-05 03:15:58.0205,2018-09-06 14:33:21.119146,97394/1/15,yellowstone-national-park,10358.0,100.0,e43ada67-8b81-4685-84a1-585edb994405,23508504.0
55293,68542,Yosemite National Park,"Yosemite National Park, US, CA",NaN,37.848335,-119.556991,37.494700,-119.886282,38.185139,-119.196421,100.0,0.476306,2014-11-12 18:01:53.245462,2018-09-06 14:33:21.681536,97394/1/14,yosemite-national-park-ca-us,10358.0,100.0,ee6016be-cb5c-4436-b84a-b46dc39bf949,55813396.0


In [29]:
def fetch_with_retry(url, max_retries=5):
    """Fetch a URL with exponential backoff on failure."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                return response.json()
            else:
                print(f'  Status {response.status_code}, retrying...')
        except (requests.exceptions.ConnectionError, 
                requests.exceptions.Timeout) as e:
            wait = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
            print(f'  Connection error: {e}. Waiting {wait}s...')
            time.sleep(wait)
    print(f'  Failed after {max_retries} attempts: {url}')
    return None

start_time = time.time()
base = 'https://api.inaturalist.org/v1/observations?place_id='
park_species = []

cols = {
    'id':                          'obs_id',
    'uuid':                        'uuid',
    'observed_on':                 'observed_on',
    'quality_grade':               'quality_grade',
    'place_guess':                 'place_guess',
    'location':                    'location',
    'captive':                     'captive',
    'taxon.name':                  'ScientificName',
    'taxon.preferred_common_name': 'CommonName',
    'taxon.rank':                  'TaxonRank',
    'taxon.iconic_taxon_name':     'TaxonGroup',
    'taxon.native':                'native',
    'taxon.introduced':            'introduced',
    'user.login':                  'user',
}

for i in range(len(poi_failed)):
    place_id = poi_failed.iloc[i]['id']
    place_name = poi_failed.iloc[i]['display_name']

    all_pages = []
    last_id = 0

    while True:
        url = f'{base}{place_id}&per_page=200&order=asc&order_by=id&id_above={last_id}'
        data = fetch_with_retry(url)

        if data is None or 'results' not in data or len(data['results']) == 0:
            break

        results = data['results']

        df = pd.json_normalize(results)
        existing_cols = {k: v for k, v in cols.items() if k in df.columns}
        df = df[existing_cols.keys()].rename(columns=existing_cols)

        if 'location' in df.columns:
            df[['latitude', 'longitude']] = df['location'].str.split(',', expand=True).astype(float)
            df = df.drop(columns='location')

        df['park_name'] = place_name
        all_pages.append(df)

        last_id = results[-1]['id']
        time.sleep(1.5)  # slightly longer sleep to be safer

    park_df = pd.concat(all_pages, ignore_index=True) if all_pages else pd.DataFrame()
    park_species.append(park_df)

    elapsed = (time.time() - start_time) / 60
    print(f'[{i+1}/{len(poi_failed)}] {place_name} — {len(park_df)} records | Elapsed: {elapsed:.1f} min')

iNaturalist2 = pd.concat(park_species, ignore_index=True)
print(f'Total records: {len(iNaturalist2)}')

  Connection error: HTTPSConnectionPool(host='api.inaturalist.org', port=443): Max retries exceeded with url: /v1/observations?place_id=4504&per_page=200&order=asc&order_by=id&id_above=247026878 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.inaturalist.org', port=443) at 0x12659c910>, 'Connection to api.inaturalist.org timed out. (connect timeout=30)')). Waiting 1s...
[1/119] Death Valley National Park, US, CA — 82942 records | Elapsed: 24.4 min
[2/119] Devils Postpile National Monument, US, CA — 1077 records | Elapsed: 24.7 min
[3/119] Devils Tower National Monument, US, WY — 4896 records | Elapsed: 26.1 min
[4/119] Dinosaur National Monument, US, CO — 8723 records | Elapsed: 28.5 min
[5/119] Eisenhower National Historic Site, US, PA — 302 records | Elapsed: 28.6 min
[6/119] El Malpais National Monument, US, NM — 3702 records | Elapsed: 29.7 min
[7/119] El Morro National Monument, US, NM — 1152 records | Elapsed: 30.0 min
[8/119] Everglades National Park, US, FL — 188212 r

In [33]:
failed_parks = [
    'Grand Portage National Monument, US, MN',
    'Grand Teton National Park, US, WY',
    'Great Basin National Park, US, NV',
    'Guadalupe Mountains National Park, US, TX',
    'Guilford Courthouse National Military Park, US, NC',
    'Gulf Islands National Seashore, US, FL',
    'Haleakala National Park, US, HI',
    'Hawaii Volcanoes National Park, US, HI',
    'Homestead National Historical Park, US, NE',
    'Hopewell Culture National Historical Park, US, OH',
    'Hopewell Furnace National Historic Site, US, PA',
    'Horseshoe Bend National Military Park, US, AL',
    'Hot Springs National Park, US, AR',
    'Hubbell Trading Post National Historic Site, US, AZ',
    'Jean Lafitte National Historical Park and Preserve, US, LA',
    'Jewel Cave National Monument, US, SD',
    'John Day Fossil Beds National Monument, US, OR',
    'Johnstown Flood National Memorial, US, PA',
    'Joshua Tree National Park, US, CA',
    'Kenai Fjords National Park, US, AK',
    'Kennesaw Mountain National Battlefield Park, US, GA',
    'Lake Meredith National Recreation Area, US, TX',
    'Lake Roosevelt National Recreation Area, US, WA',
    'Lassen Volcanic National Park, US, CA',
    'Lava Beds National Monument, US, CA',
    'Lewis and Clark National Historical Park, US, OR',
    'Lincoln Boyhood National Memorial, US, IN',
    'Little Bighorn Battlefield National Monument, US, MT',
    'Little River Canyon National Preserve, US, AL',
    'Lyndon B. Johnson National Historical Park, US, TX',
    'Mammoth Cave National Park, US, KY',
    'Manassas National Battlefield Park, US, VA',
    'Manzanar National Historic Site, US, CA',
    'Martin Van Buren National Historic Site, US, NY',
    'Mesa Verde National Park, US, CO',
    'Minute Man National Historical Park, US, MA',
    'Mojave National Preserve, US, CA',
    'Monocacy National Battlefield, US, MD',
    'Montezuma Castle National Monument, US, AZ',
    'Moores Creek National Battlefield, US, NC',
    'Morristown National Historical Park, US, NJ',
    'Mount Rainier National Park, US, WA',
    'Mount Rushmore National Memorial, US, SD',
    'Natural Bridges National Monument, US, UT',
    'Navajo National Monument, US, AZ',
    'New River Gorge National Park and Preserve, US, WV',
    'Nez Perce National Historical Park, US, ID',
    'North Cascades National Park, US, WA',
    'Organ Pipe Cactus National Monument, US, AZ',
    'Padre Island National Seashore, US, TX',
    'Palo Alto Battlefield National Historical Park, US, TX',
    'Pea Ridge National Military Park, US, AR',
    'Petersburg National Battlefield, US, VA',
    'Petrified Forest National Park, US, AZ',
    'Petroglyph National Monument, US, NM',
    'Pictured Rocks National Lakeshore, US, MI',
    'Pinnacles National Park, US, CA',
    'Pipe Spring National Monument, US, AZ',
    'Piscataway Park, US, MD',
    'Point Reyes National Seashore, US, CA',
    'Prince William Forest Park, US, VA',
    "Pu'ukohola Heiau National Historic Site, US, HI",
    'Redwood National Park, US, CA',
    'Richmond National Battlefield Park, US, VA',
    'Rocky Mountain National Park, US, CO',
    'Russell Cave National Monument, US, AL',
    'Saguaro National Park, US, AZ',
    'San Antonio Missions National Historical Park, US, TX',
    'Santa Monica Mountains National Recreation Area, US, CA',
    'Saratoga National Historical Park, US, NY',
    'Scotts Bluff National Monument, US, NE',
    'Shenandoah National Park, US, VA',
    'Sleeping Bear Dunes National Lakeshore, US, MI',
    'Sunset Crater Volcano National Monument, US, AZ',
    'Theodore Roosevelt National Park, US, ND',
    'Timucuan Ecological and Historic Preserve, US, FL',
    'Tonto National Monument, US, AZ',
    'Tuzigoot National Monument, US, AZ',
    'Upper Delaware Scenic and Recreational River, US, NY',
    'Valley Forge National Historical Park, US, PA',
    'Vanderbilt Mansion National Historic Site, US, NY',
    'Walnut Canyon National Monument, US, AZ',
    'White Sands National Park, US, NM',
    'Whitman Mission National Historic Site, US, WA',
    "Wilson's Creek National Battlefield, US, MO",
    'Wind Cave National Park, US, SD',
    'Wright Brothers National Memorial, US, NC',
    'Yellowstone National Park, US, WY',
    'Yosemite National Park, US, CA',
    'Zion National Park, US, UT'
]

print(f"Total failed parks: {len(failed_parks)}")

Total failed parks: 90


In [57]:
iNaturalist2.to_csv('iNaturalist2.csv', index=False)

In [35]:
poi_failed2 = poi[poi['display_name'].isin(failed_parks)]
print(f"Matched: {len(poi_failed)}")

Matched: 119


In [16]:
def scrape_iNaturalist(poi_failed):
    def fetch_with_retry(url, max_retries=5):
        """Fetch a URL with exponential backoff on failure."""
        for attempt in range(max_retries):
            try:
                response = requests.get(url, timeout=30)
                if response.status_code == 200:
                    return response.json()
                else:
                    print(f'  Status {response.status_code}, retrying...')
            except (requests.exceptions.ConnectionError, 
                    requests.exceptions.Timeout) as e:
                wait = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
                print(f'  Connection error: {e}. Waiting {wait}s...')
                time.sleep(wait)
        print(f'  Failed after {max_retries} attempts: {url}')
        return None

    start_time = time.time()
    base = 'https://api.inaturalist.org/v1/observations?place_id='
    park_species = []
    
    cols = {
        'id':                          'obs_id',
        'uuid':                        'uuid',
        'observed_on':                 'observed_on',
        'quality_grade':               'quality_grade',
        'place_guess':                 'place_guess',
        'location':                    'location',
        'captive':                     'captive',
        'taxon.name':                  'ScientificName',
        'taxon.preferred_common_name': 'CommonName',
        'taxon.rank':                  'TaxonRank',
        'taxon.iconic_taxon_name':     'TaxonGroup',
        'taxon.native':                'native',
        'taxon.introduced':            'introduced',
        'user.login':                  'user',
    }
    
    for i in range(len(poi_failed)):
        place_id = poi_failed.iloc[i]['id']
        place_name = poi_failed.iloc[i]['display_name']
    
        all_pages = []
        last_id = 0
    
        while True:
            url = f'{base}{place_id}&per_page=200&order=asc&order_by=id&id_above={last_id}'
            data = fetch_with_retry(url)
    
            if data is None or 'results' not in data or len(data['results']) == 0:
                break
    
            results = data['results']
    
            df = pd.json_normalize(results)
            existing_cols = {k: v for k, v in cols.items() if k in df.columns}
            df = df[existing_cols.keys()].rename(columns=existing_cols)
    
            if 'location' in df.columns:
                df[['latitude', 'longitude']] = df['location'].str.split(',', expand=True).astype(float)
                df = df.drop(columns='location')
    
            df['park_name'] = place_name
            all_pages.append(df)
    
            last_id = results[-1]['id']
            time.sleep(1.5)  # slightly longer sleep to be safer
    
        park_df = pd.concat(all_pages, ignore_index=True) if all_pages else pd.DataFrame()
        park_species.append(park_df)
    
        elapsed = (time.time() - start_time) / 60
        print(f'[{i+1}/{len(poi_failed)}] {place_name} — {len(park_df)} records | Elapsed: {elapsed:.1f} min')
    
    iNaturalist2 = pd.concat(park_species, ignore_index=True)
    iNaturalist2.to_csv('iNaturalist2.csv', index=False)

In [55]:
scrape_iNaturalist(poi_failed2)

[1/91] Grand Portage National Monument, US, MN — 860 records | Elapsed: 0.3 min
[2/91] Grand Teton National Park, US, WY — 69915 records | Elapsed: 21.5 min
[3/91] Great Basin National Park, US, NV — 18375 records | Elapsed: 26.7 min
[4/91] Guadalupe Mountains National Park, US, TX — 24645 records | Elapsed: 33.6 min
[5/91] Guilford Courthouse National Military Park, US, NC — 1483 records | Elapsed: 34.1 min
[6/91] Gulf Islands National Seashore, US, FL — 40022 records | Elapsed: 47.0 min
[7/91] Haleakala National Park, US, HI — 11647 records | Elapsed: 50.7 min
[8/91] Hawaii Volcanoes National Park, US, HI — 41162 records | Elapsed: 63.3 min
[9/91] Homestead National Historical Park, US, NE — 2016 records | Elapsed: 64.1 min
[10/91] Hopewell Culture National Historical Park, US, OH — 1075 records | Elapsed: 64.5 min
[11/91] Hopewell Furnace National Historic Site, US, PA — 2160 records | Elapsed: 65.5 min
[12/91] Horseshoe Bend National Military Park, US, AL — 395 records | Elapsed: 6

In [59]:
failed3 = ['Mesa Verde National Park, US, CO',
    'Minute Man National Historical Park, US, MA',
    'Mojave National Preserve, US, CA',
    'Monocacy National Battlefield, US, MD',
    'Montezuma Castle National Monument, US, AZ',
    'Moores Creek National Battlefield, US, NC',
    'Morristown National Historical Park, US, NJ',
    'Mount Rainier National Park, US, WA',
    'Mount Rushmore National Memorial, US, SD',
    'Natural Bridges National Monument, US, UT',
    'Navajo National Monument, US, AZ',
    'New River Gorge National Park and Preserve, US, WV',
    'Nez Perce National Historical Park, US, ID',
    'North Cascades National Park, US, WA',
    'Organ Pipe Cactus National Monument, US, AZ',
    'Padre Island National Seashore, US, TX',
    'Palo Alto Battlefield National Historical Park, US, TX',
    'Pea Ridge National Military Park, US, AR',
    'Petersburg National Battlefield, US, VA',
    'Petrified Forest National Park, US, AZ',
    'Petroglyph National Monument, US, NM',
    'Pictured Rocks National Lakeshore, US, MI',
    'Pinnacles National Park, US, CA',
    'Pipe Spring National Monument, US, AZ',
    'Piscataway Park, US, MD',
    'Point Reyes National Seashore, US, CA',
    'Prince William Forest Park, US, VA',
    "Pu'ukohola Heiau National Historic Site, US, HI",
    'Redwood National Park, US, CA',
    'Richmond National Battlefield Park, US, VA',
    'Rocky Mountain National Park, US, CO',
    'Russell Cave National Monument, US, AL',
    'Saguaro National Park, US, AZ',
    'San Antonio Missions National Historical Park, US, TX',
    'Santa Monica Mountains National Recreation Area, US, CA',
    'Saratoga National Historical Park, US, NY',
    'Scotts Bluff National Monument, US, NE',
    'Shenandoah National Park, US, VA',
    'Sleeping Bear Dunes National Lakeshore, US, MI',
    'Sunset Crater Volcano National Monument, US, AZ',
    'Theodore Roosevelt National Park, US, ND',
    'Timucuan Ecological and Historic Preserve, US, FL',
    'Tonto National Monument, US, AZ',
    'Tuzigoot National Monument, US, AZ',
    'Upper Delaware Scenic and Recreational River, US, NY',
    'Valley Forge National Historical Park, US, PA',
    'Vanderbilt Mansion National Historic Site, US, NY',
    'Walnut Canyon National Monument, US, AZ',
    'White Sands National Park, US, NM',
    'Whitman Mission National Historic Site, US, WA',
    "Wilson's Creek National Battlefield, US, MO",
    'Wind Cave National Park, US, SD',
    'Wright Brothers National Memorial, US, NC',
    'Yellowstone National Park, US, WY',
    'Yosemite National Park, US, CA',
    'Zion National Park, US, UT']

In [65]:
poi_failed3 = poi[poi['display_name'].isin(failed3)]
scrape_iNaturalist(poi_failed3)

[1/56] Mesa Verde National Park, US, CO — 10490 records | Elapsed: 3.0 min
[2/56] Minute Man National Historical Park, US, MA — 11703 records | Elapsed: 7.2 min
[3/56] Mojave National Preserve, US, CA — 60135 records | Elapsed: 25.2 min
[4/56] Monocacy National Battlefield, US, MD — 3832 records | Elapsed: 26.6 min
[5/56] Montezuma Castle National Monument, US, AZ — 5329 records | Elapsed: 28.1 min
[6/56] Moores Creek National Battlefield, US, NC — 610 records | Elapsed: 28.3 min
[7/56] Morristown National Historical Park, US, NJ — 5234 records | Elapsed: 30.0 min
[8/56] Mount Rainier National Park, US, WA — 90207 records | Elapsed: 55.9 min
[9/56] Mount Rushmore National Memorial, US, SD — 1812 records | Elapsed: 56.5 min
[10/56] Natural Bridges National Monument, US, UT — 2048 records | Elapsed: 57.1 min
[11/56] Navajo National Monument, US, AZ — 116 records | Elapsed: 57.2 min
[12/56] New River Gorge National Park and Preserve, US, WV — 32924 records | Elapsed: 67.0 min
[13/56] Nez 

In [18]:
failed4 = ['Redwood National Park, US, CA',
    'Richmond National Battlefield Park, US, VA',
    'Rocky Mountain National Park, US, CO',
    'Russell Cave National Monument, US, AL',
    'Saguaro National Park, US, AZ',
    'San Antonio Missions National Historical Park, US, TX',
    'Santa Monica Mountains National Recreation Area, US, CA',
    'Saratoga National Historical Park, US, NY',
    'Scotts Bluff National Monument, US, NE',
    'Shenandoah National Park, US, VA',
    'Sleeping Bear Dunes National Lakeshore, US, MI',
    'Sunset Crater Volcano National Monument, US, AZ',
    'Theodore Roosevelt National Park, US, ND',
    'Timucuan Ecological and Historic Preserve, US, FL',
    'Tonto National Monument, US, AZ',
    'Tuzigoot National Monument, US, AZ',
    'Upper Delaware Scenic and Recreational River, US, NY',
    'Valley Forge National Historical Park, US, PA',
    'Vanderbilt Mansion National Historic Site, US, NY',
    'Walnut Canyon National Monument, US, AZ',
    'White Sands National Park, US, NM',
    'Whitman Mission National Historic Site, US, WA',
    "Wilson's Creek National Battlefield, US, MO",
    'Wind Cave National Park, US, SD',
    'Wright Brothers National Memorial, US, NC',
    'Yellowstone National Park, US, WY',
    'Yosemite National Park, US, CA',
    'Zion National Park, US, UT']

In [20]:
poi_failed4 = poi[poi['display_name'].isin(failed4)]
scrape_iNaturalist(poi_failed4)

[1/28] Redwood National Park, US, CA — 102208 records | Elapsed: 33.6 min
[2/28] Richmond National Battlefield Park, US, VA — 6206 records | Elapsed: 35.8 min
[3/28] Rocky Mountain National Park, US, CO — 106270 records | Elapsed: 70.9 min
[4/28] Russell Cave National Monument, US, AL — 1348 records | Elapsed: 71.4 min
[5/28] Saguaro National Park, US, AZ — 80754 records | Elapsed: 96.9 min
[6/28] San Antonio Missions National Historical Park, US, TX — 12310 records | Elapsed: 101.2 min
[7/28] Santa Monica Mountains National Recreation Area, US, CA — 339059 records | Elapsed: 219.1 min
[8/28] Saratoga National Historical Park, US, NY — 1873 records | Elapsed: 219.9 min
[9/28] Scotts Bluff National Monument, US, NE — 2259 records | Elapsed: 220.6 min
[10/28] Shenandoah National Park, US, VA — 119073 records | Elapsed: 259.3 min
[11/28] Sleeping Bear Dunes National Lakeshore, US, MI — 29100 records | Elapsed: 269.3 min
[12/28] Sunset Crater Volcano National Monument, US, AZ — 1844 record

In [26]:
failed5 = ['Zion National Park, US, UT']
poi_failed5 = poi[poi['display_name'].isin(failed5)]
scrape_iNaturalist(poi_failed5)

[1/1] Zion National Park, US, UT — 63091 records | Elapsed: 17.9 min


In [30]:
A = pd.read_csv('iNat_A-D.csv')
D = pd.read_csv('iNat_D-G.csv', low_memory=False)
G = pd.read_csv('iNat_G-M.csv')
M = pd.read_csv('iNat_M-P.csv')
P = pd.read_csv('iNat_P-Y.csv')
Z = pd.read_csv('iNat_zion.csv')
X = pd.read_csv('iNaturalist2.csv')

In [34]:
iNaturalist = pd.concat([A, D, G, M, P, Z, X], ignore_index=True)
iNaturalist

,obs_id,uuid,observed_on,quality_grade,place_guess,captive,ScientificName,CommonName,TaxonRank,TaxonGroup,native,introduced,user,latitude,longitude,park_name
0,2634307,76de9be0-be63-4eba-bbbb-97b9a06bd11e,2016-01-31,needs_id,"Abraham Lincoln Birthplace, Hodgenville, KY, US",False,Vertebrata,Vertebrates,subphylum,Animalia,False,False,deb6,37.532783,-85.732837,Abraham Lincoln Birthplace National Historical...
1,4427636,f92bd7dc-58eb-444e-a267-02a7052286e7,2016-10-24,research,"Pathway of a President Hodgenville, KY 42748",False,Cerioporus squamosus,Dryad's Saddle,species,Fungi,True,False,rambler,37.530714,-85.735752,Abraham Lincoln Birthplace National Historical...
2,5626015,245de8a2-62da-4739-9770-a939ba33edbd,2017-04-09,research,Abraham Lincoln Birthplace National Historical...,False,Podophyllum peltatum,mayapple,species,Plantae,True,False,nbaumbach,37.530752,-85.738697,Abraham Lincoln Birthplace National Historical...
3,6134510,4fdd05b3-ea0b-481e-b839-d66133bd8c8e,2015-06-29,research,"2995 Lincoln Farm Rd, Hodgenville, KY 42748, USA",False,Limenitis arthemis astyanax,Red-spotted Purple,subspecies,Insecta,True,False,vakapi,37.531128,-85.737520,Abraham Lincoln Birthplace National Historical...
4,7983771,8560552f-9e69-4b71-bc61-a63562079302,2017-09-18,research,"Abraham Lincoln's Boyhood Home at Knob Creek, ...",False,Arnoglossum reniforme,Great Indian Plantain,species,Plantae,True,False,milopyne,37.614610,-85.645981,Abraham Lincoln Birthplace National Historical...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4104604,352971810,6cb87f84-0f43-4bfb-9d8e-752619383ca6,2026-04-23,research,"ZION NATIONAL PARK, UT 84767, USA",False,Otospermophilus variegatus,Rock Squirrel,species,Mammalia,True,False,tinaboville,37.295689,-112.947881,"Zion National Park, US, UT"
4104605,352972045,e65ea6a5-20b8-4e9f-a3ed-244186c45093,2026-04-23,needs_id,"ZION NATIONAL PARK, UT 84767, USA",False,Cinclus mexicanus,American Dipper,species,Aves,True,False,tinaboville,37.294519,-112.948161,"Zion National Park, US, UT"
4104606,352998704,cfeb1a00-3466-4915-9d5a-066b964b8994,2026-04-23,needs_id,"Zion National Park, Springdale, UT, US",False,Uropappus lindleyi,silverpuffs,species,Plantae,True,False,dustyapril,37.204392,-112.980513,"Zion National Park, US, UT"
4104607,352999736,89d02c86-be23-4453-a906-9cf355b7188a,2026-04-23,needs_id,"Zion National Park, Springdale, UT, US",False,Aciurina bigeloviae,Cotton-gall Tephritid,species,Insecta,True,False,dustyapril,37.204195,-112.980658,"Zion National Park, US, UT"


In [40]:
print(f'Before: {len(iNaturalist)}')
iNaturalist = iNaturalist.drop_duplicates(subset='obs_id')
print(f'After: {len(iNaturalist)}')

Before: 4033633
After: 4033633


In [744]:
# calculates shannon diversity index
def shannon_diversity(group):
    # count observations per species
    species_counts = group['ScientificName'].value_counts()
    total = species_counts.sum()
    proportions = species_counts / total
    H = -(proportions * np.log(proportions)).sum()
    return H

sdi = (iNaturalist
       .groupby('park_name')
       .apply(shannon_diversity)
       .reset_index()
       .rename(columns={0: 'SDI'}))

# creates counts for observations and species
park_stats = iNaturalist.groupby('park_name').agg(
    n_observations = ('obs_id', 'count'),
    n_species =      ('ScientificName', 'nunique'),
).reset_index()

sdi = sdi.merge(park_stats, on='park_name')
sdi

,park_name,SDI,n_observations,n_species
0,Abraham Lincoln Birthplace National Historical...,5.993603,816,500
1,"Acadia National Park, US, ME",7.061506,109130,6472
2,Allegheny Portage Railroad National Historic S...,6.201320,1359,698
3,"Amistad National Recreation Area, US, TX",6.939610,10331,2268
4,"Apostle Islands National Lakeshore, US, WI",6.492174,4434,1372
...,...,...,...,...
147,"Wind Cave National Park, US, SD",6.164829,9954,1508
148,"Wright Brothers National Memorial, US, NC",5.366586,746,361
149,"Yellowstone National Park, US, WY",6.459104,117750,4941
150,"Yosemite National Park, US, CA",6.973596,101600,5540


In [81]:
sdi.to_csv('SDI.csv', index=False)

In [52]:
# calculates slope for observations and species over time
import numpy as np
from scipy import stats

# convert to datetime and extract year
iNaturalist['observed_on'] = pd.to_datetime(iNaturalist['observed_on'])
iNaturalist['year'] = iNaturalist['observed_on'].dt.year

# aggregate per park per year
yearly = iNaturalist.groupby(['park_name', 'year']).agg(
    n_observations = ('obs_id', 'count'),
    n_species =      ('ScientificName', 'nunique'),
).reset_index()

# calculate slope for each park
def get_slopes(group):
    if len(group) < 3:  # need at least 3 years for a meaningful slope
        return pd.Series({'obs_slope': np.nan, 'species_slope': np.nan})
    
    obs_slope, _, _, _, _       = stats.linregress(group['year'], group['n_observations'])
    species_slope, _, _, _, _   = stats.linregress(group['year'], group['n_species'])
    
    return pd.Series({
        'obs_slope':     obs_slope,
        'species_slope': species_slope,
    })

slopes = (yearly
          .groupby('park_name')
          .apply(get_slopes)
          .reset_index())

print(slopes.shape)
print(slopes.head())

(152, 3)
                                           park_name   obs_slope  \
0  Abraham Lincoln Birthplace National Historical...   14.062937   
1                       Acadia National Park, US, ME  225.661010   
2  Allegheny Portage Railroad National Historic S...   39.127273   
3           Amistad National Recreation Area, US, TX   23.278568   
4         Apostle Islands National Lakeshore, US, WI   41.925059   

   species_slope  
0      10.884615  
1      40.523488  
2      24.284848  
3      10.157715  
4      24.379651  


# RESPONSE VARIABLES

In [702]:
# these would have to be predictors...right? because the SDI is calculated from these values so they would be correlated
# could do multi-label prediciton?
animals = sdi.merge(slopes, on = 'park_name')


In [65]:
def get_nasa_climate(lat, lon, start='2010', end='2020'):
    url = (f'https://power.larc.nasa.gov/api/temporal/climatology/point'
           f'?parameters=T2M,PRECTOTCORR,RH2M'
           f'&community=RE&longitude={lon}&latitude={lat}&format=JSON')
    response = requests.get(url)
    data = response.json()
    params = data['properties']['parameter']
    return {
        'avg_temp':  np.mean(list(params['T2M'].values())),
        'avg_precip': np.mean(list(params['PRECTOTCORR'].values())),
        'avg_humidity': np.mean(list(params['RH2M'].values())),
    }

climate_records = []
for idx, row in poi.iterrows():
    climate = get_nasa_climate(row['latitude'], row['longitude'])
    climate['park_name'] = row['display_name']
    climate_records.append(climate)
    print(f"Fetched {row['display_name']}")
    time.sleep(0.5)  # be polite to the API

climate_df = pd.DataFrame(climate_records)

Fetched Abraham Lincoln Birthplace National Historical Park, US, KY
Fetched Acadia National Park, US, ME
Fetched SERC Education - Schoodic - Acadia National Park, US, ME
Fetched Allegheny Portage Railroad National Historic Site, US, PA
Fetched Amistad National Recreation Area, US, TX
Fetched Apostle Islands National Lakeshore, US, WI
Fetched Appomattox Court House National Historical Park, US, VA
Fetched Arches National Park, US, UT
Fetched Arkansas Post National Memorial, US, AR
Fetched Badlands National Park, US, SD
Fetched Big Bend National Park, US, TX
Fetched Big Cypress National Preserve, US, FL
Fetched Big Hole National Battlefield, US, MT
Fetched Big Thicket National Preserve, US, TX
Fetched Bighorn Canyon National Recreation Area, US, MT
Fetched Biscayne National Park, US, FL
Fetched Black Canyon of the Gunnison National Park, US, CO
Fetched Booker T. Washington National Monument, US, VA
Fetched Bryce Canyon National Park, US, UT
Fetched Buffalo National River, US, AR
Fetched 

In [83]:
climate_df.to_csv('climate.csv', index=False)

In [75]:
url = (f'https://power.larc.nasa.gov/api/temporal/climatology/point'
           f'?parameters=T2M,PRECTOTCORR,RH2M'
           f'&community=RE&longitude=37.532783&latitude=-85.732837&format=JSON')
response = requests.get(url)
data = response.json()
params = data['properties']['parameter']

In [85]:
census_url = 'https://api.census.gov/data/2020/dec/pl?get=NAME,P1_001N&for=county:*'
response = requests.get(census_url)
census_df = pd.DataFrame(response.json()[1:], columns=response.json()[0])

,NAME,county_id,state,county
0,"Autauga County, Alabama",58805,01,001
1,"Baldwin County, Alabama",231767,01,003
2,"Barbour County, Alabama",25223,01,005
3,"Bibb County, Alabama",22293,01,007
4,"Blount County, Alabama",59134,01,009
...,...,...,...,...
3216,"Renville County, Minnesota",14723,27,129
3217,"Roseau County, Minnesota",15331,27,135
3218,"Sherburne County, Minnesota",97183,27,141
3219,"Steele County, Minnesota",37406,27,147


In [97]:
def get_county(lat, lon):
    url = f'https://geo.fcc.gov/api/census/block/find?latitude={lat}&longitude={lon}&format=json'
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        return {
            'county':      data['County']['name'],
            'county_fips': data['County']['FIPS'],
            'state':       data['State']['name'],
            'state_fips':  data['State']['FIPS'],
        }
    except Exception as e:
        print(f'Error for {lat}, {lon}: {e}')
        return {'county': None, 'county_fips': None, 'state': None, 'state_fips': None}

# use your nps_df which already has lat/lon
county_records = []
for idx, row in poi.iterrows():
    result = get_county(row['latitude'], row['longitude'])
    result['park_code'] = row['id']
    result['park_name'] = row['display_name']
    county_records.append(result)
    print(f"[{idx+1}/{len(poi)}] {row['display_name']} → {result['county']}")
    time.sleep(0.5)

county_df = pd.DataFrame(county_records)
print(county_df.head())

[71060/170] Abraham Lincoln Birthplace National Historical Park, US, KY → Larue County
[50093/170] Acadia National Park, US, ME → Hancock County
[83344/170] SERC Education - Schoodic - Acadia National Park, US, ME → Hancock County
[59821/170] Allegheny Portage Railroad National Historic Site, US, PA → Cambria County
[71069/170] Amistad National Recreation Area, US, TX → Val Verde County
[71078/170] Apostle Islands National Lakeshore, US, WI → Ashland County
[71077/170] Appomattox Court House National Historical Park, US, VA → Appomattox County
[52852/170] Arches National Park, US, UT → Grand County
[71079/170] Arkansas Post National Memorial, US, AR → Arkansas County
[59822/170] Badlands National Park, US, SD → Oglala Lakota County
[52853/170] Big Bend National Park, US, TX → Brewster County
[71087/170] Big Cypress National Preserve, US, FL → Collier County
[71090/170] Big Hole National Battlefield, US, MT → Beaverhead County
[71093/170] Big Thicket National Preserve, US, TX → Hardin C

In [133]:
census_df = census_df.rename(columns = {"county_id": "population"})
county_df = county_df.rename(columns = {"county_fips": "county_id"})

In [121]:
census_df.merge(county_df, on = "county_id", how = "inner")

,NAME,county_id,state_x,county_x,county_y,state_y,state_fips,park_code,park_name
0,"Meade County, Kentucky",30003,21,163,Big Horn County,Montana,30,95102,"Bighorn Canyon National Recreation Area, US, MT"
1,"Meade County, Kentucky",30003,21,163,Big Horn County,Montana,30,95261,"Little Bighorn Battlefield National Monument, ..."
2,"Cass County, Iowa",13127,19,029,Glynn County,Georgia,13,95184,"Fort Frederica National Monument, US, GA"
3,"Boyd County, Kentucky",48261,21,019,Kenedy County,Texas,48,95300,"Padre Island National Seashore, US, TX"


In [135]:
census_df['fips'] = census_df['state'].str.zfill(2) + census_df['county'].str.zfill(3)
county_df['fips'] = county_df['county_id'].astype(str).str.zfill(5)
park_census = county_df.merge(census_df[['fips', 'population']], on='fips', how='left')

In [137]:
park_census

,county,county_id,state,state_fips,park_code,park_name,fips,population
0,Larue County,21123,Kentucky,21,90763,Abraham Lincoln Birthplace National Historical...,21123,14867
1,Hancock County,23009,Maine,23,49610,"Acadia National Park, US, ME",23009,55478
2,Hancock County,23009,Maine,23,117673,SERC Education - Schoodic - Acadia National Pa...,23009,55478
3,Cambria County,42021,Pennsylvania,42,72632,Allegheny Portage Railroad National Historic S...,42021,133472
4,Val Verde County,48465,Texas,48,95085,"Amistad National Recreation Area, US, TX",48465,47586
...,...,...,...,...,...,...,...,...
165,Custer County,46033,South Dakota,46,72794,"Wind Cave National Park, US, SD",46033,8318
166,Dare County,37055,North Carolina,37,95344,"Wright Brothers National Memorial, US, NC",37055,36915
167,Park County,56029,Wyoming,56,10211,"Yellowstone National Park, US, WY",56029,29624
168,Mariposa County,06043,California,06,68542,"Yosemite National Park, US, CA",06043,17131


In [153]:
# use population density instead
# check what variables are available first
area_url = "https://api.census.gov/data/2024/geoinfo?get=NAME,AREALAND&for=county:*"
response = requests.get(area_url)
data = response.json()

area_df = pd.DataFrame(data[1:], columns=data[0])

In [161]:
area_df["AREALAND"] = area_df["AREALAND"].astype(float)/1e6

In [167]:
census = census_df.merge(area_df, on = "NAME")
census['pop_density'] = census['population'].astype(float)/census['AREALAND'].astype(float)

In [285]:
# population density in people/km
park_census = county_df.merge(census[['fips', 'population', 'pop_density', "county_x"]], on='fips', how='left')

In [287]:
park_census

,county,county_id,state,state_fips,park_code,park_name,fips,population,pop_density,county_x
0,Larue County,21123,Kentucky,21,90763,Abraham Lincoln Birthplace National Historical...,21123,14867,21.989644,123
1,Hancock County,23009,Maine,23,49610,"Acadia National Park, US, ME",23009,55478,13.496177,009
2,Hancock County,23009,Maine,23,117673,SERC Education - Schoodic - Acadia National Pa...,23009,55478,13.496177,009
3,Cambria County,42021,Pennsylvania,42,72632,Allegheny Portage Railroad National Historic S...,42021,133472,74.907834,021
4,Val Verde County,48465,Texas,48,95085,"Amistad National Recreation Area, US, TX",48465,47586,5.842455,465
...,...,...,...,...,...,...,...,...,...,...
165,Custer County,46033,South Dakota,46,72794,"Wind Cave National Park, US, SD",46033,8318,2.062759,033
166,Dare County,37055,North Carolina,37,95344,"Wright Brothers National Memorial, US, NC",37055,36915,37.191761,055
167,Park County,56029,Wyoming,56,10211,"Yellowstone National Park, US, WY",56029,29624,1.648339,029
168,Mariposa County,06043,California,06,68542,"Yosemite National Park, US, CA",06043,17131,4.565251,043


In [277]:
air = pd.read_csv('DATA/airQualityData.csv', header = 2)
air = air.dropna(subset=['County FIPS Code'])
air['fips'] = air['County FIPS Code'].astype(int).astype(str).str.zfill(5)

In [291]:
# dataset containing air quality data and population data
air_population = pd.merge(park_census, air, how = "left", on = "fips")

### Toxic Release Inventory

In [296]:
TRI = pd.read_csv('DATA/ToxicReleaseInventory.csv')

/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_80408/3785251288.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  TRI = pd.read_csv('DATA/ToxicReleaseInventory.csv')


In [334]:
TRI.columns[106]

'107. TOTAL RELEASES'

In [336]:
def haversine(lat1, lon1, lat2, lon2):
    """distance in km between two lat/lon points"""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_tri_features(park_lat, park_lon, tri_df, radius_km=50):
    """
    For a given park, find all TRI facilities within radius_km
    and compute pollution features
    """
    tri_df = tri_df.copy()
    
    # compute distance from park to each facility
    tri_df['distance_km'] = tri_df.apply(
        lambda r: haversine(park_lat, park_lon, r['12. LATITUDE'], r['13. LONGITUDE']), axis=1
    )
    
    # filter to radius
    nearby = tri_df[tri_df['distance_km'] <= radius_km]
    
    if len(nearby) == 0:
        return {
            'n_facilities':        0,
            'total_releases':      0,
            'weighted_pollution':  0,
            'max_release':         0,
        }
    
    # inverse distance weighting — closer facilities count more
    nearby['weight'] = 1 / (nearby['distance_km'] + 1)  # +1 avoids division by zero
    nearby['weighted_release'] = nearby['weight'] * nearby['107. TOTAL RELEASES']
    
    return {
        'n_facilities':       len(nearby),
        'total_releases':     nearby['107. TOTAL RELEASES'].sum(),
        'weighted_pollution': nearby['weighted_release'].sum(),
        'max_release':        nearby['107. TOTAL RELEASES'].max(),
    }

In [338]:
tri_records = []
for idx, row in poi.iterrows():
    features = get_tri_features(row['latitude'], row['longitude'], TRI, radius_km=50)
    features['park_name'] = row['display_name']
    tri_records.append(features)

tri_features = pd.DataFrame(tri_records)

/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_80408/1697735434.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nearby['weight'] = 1 / (nearby['distance_km'] + 1)  # +1 avoids division by zero
/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_80408/1697735434.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nearby['weighted_release'] = nearby['weight'] * nearby['107. TOTAL RELEASES']
/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_80408/1697735434.py:34: SettingWithC

In [349]:
# dataset with total chemical release from plants
with_TRI = pd.merge(air_population, tri_features, on = "park_name")
with_TRI.head(5)

,county,county_id,state,state_fips,park_code,park_name,fips,population,pop_density,county_x,...,O3 8-hr (ppm),PM10 24-hr (µg/m3),PM2.5 Wtd AM (µg/m3),PM2.5 24-hr (µg/m3),SO2 1-hr (ppb),SO2 Wtd AM (ppb),n_facilities,total_releases,weighted_pollution,max_release
0,Larue County,21123,Kentucky,21,90763,Abraham Lincoln Birthplace National Historical...,21123,14867,21.989644,123,...,NaN,NaN,NaN,NaN,NaN,NaN,61,717694.692,21084.846952,291540.0
1,Hancock County,23009,Maine,23,49610,"Acadia National Park, US, ME",23009,55478,13.496177,009,...,0.069,24,3.5,12,1,0,1,5405.000,309.399860,5405.0
2,Hancock County,23009,Maine,23,117673,SERC Education - Schoodic - Acadia National Pa...,23009,55478,13.496177,009,...,0.069,24,3.5,12,1,0,1,5405.000,191.381867,5405.0
3,Cambria County,42021,Pennsylvania,42,72632,Allegheny Portage Railroad National Historic S...,42021,133472,74.907834,021,...,0.061,37,6.6,15,11,1,125,3231842.377,125045.981823,703103.0
4,Val Verde County,48465,Texas,48,95085,"Amistad National Recreation Area, US, TX",48465,47586,5.842455,465,...,NaN,NaN,NaN,NaN,NaN,NaN,6,10257.165,318.452007,6950.0


### Light Pollution

In [367]:

# Step 1: get a NASA Earthdata token
# register free at https://urs.earthdata.nasa.gov/users/new
# then:

EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'nps_night_lights',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'VNP46A4.002', 'layer': 'NearNadir_Composite_Snow_Free'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

# Step 4: poll until done (takes a few minutes)
while True:
    status = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
        headers=headers
    ).json()['status']
    print(f'Status: {status}')
    if status == 'done':
        break
    time.sleep(30)

# Step 5: download results
files = requests.get(
    f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
    headers=headers
).json()['files']

for f in files:
    if f['file_name'].endswith('.csv'):
        dl = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
            headers=headers,
            stream=True
        )
        with open(f['file_name'], 'wb') as out:
            for chunk in dl.iter_content(chunk_size=8192):
                out.write(chunk)
        print(f'Downloaded: {f["file_name"]}')

Task submitted: {'message': 'Product VNP46A4.002 is not available'}


KeyError: 'status'

In [445]:
products = requests.get('https://appeears.earthdatacloud.nasa.gov/api/product').json()

df_products = pd.DataFrame(products)
print(df_products[['Description', 'ProductAndVersion']].to_string())

                                                                          Description                            ProductAndVersion
0                                                                     Land Cover Type                                  MCD12Q1.061
1                                                                 Land Cover Dynamics                                  MCD12Q2.061
2    Leaf Area Index (LAI) and Fraction of Photosynthetically Active Radiation (FPAR)                                 MCD15A2H.061
3    Leaf Area Index (LAI) and Fraction of Photosynthetically Active Radiation (FPAR)                                 MCD15A3H.061
4                   Bidirectional Reflectance Distribution Function (BRDF) and Albedo                                  MCD43A1.061
5                   Bidirectional Reflectance Distribution Function (BRDF) and Albedo                                  MCD43A2.061
6                   Bidirectional Reflectance Distribution Function (BRDF) and Albe

In [379]:
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'nps_night_lights',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MCD12Q1.061', 'layer': 'LC_Type1'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': '3b10f138-2907-4d99-b328-91d005d640eb', 'status': 'pending'}


In [387]:
task_id = resp.json()['task_id']  # extract just the id string

while True:
    status = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
        headers=headers
    ).json()['status']
    print(f'Status: {status}')
    if status == 'done':
        break
    time.sleep(30)

# Step 5: download results
files = requests.get(
    f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
    headers=headers
).json()['files']

for f in files:
    if f['file_name'].endswith('.csv'):
        dl = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
            headers=headers,
            stream=True
        )
        with open(f['file_name'], 'wb') as out:
            for chunk in dl.iter_content(chunk_size=8192):
                out.write(chunk)
        print(f'Downloaded: {f["file_name"]}')

Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: done
Downloaded: nps-night-lights-MCD12Q1-061-results.csv


In [389]:
def NASAapi(product, layer):
    EARTHDATA_USER = 'isabellawoods'
    EARTHDATA_PASS = 'Atlanta!1101'
    
    # get token
    token_resp = requests.post(
        'https://appeears.earthdatacloud.nasa.gov/api/login',
        auth=(EARTHDATA_USER, EARTHDATA_PASS)
    )
    token = token_resp.json()['token']
    headers = {'Authorization': f'Bearer {token}'}
    
    # Step 2: submit a point sample task for all parks
    task = {
        'task_type': 'point',
        'task_name': 'nps_night_lights',
        'params': {
            'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
            'layers': [{'product': 'MCD12Q1.061', 'layer': 'LC_Type1'}],
            'coordinates': [
                {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
                for _, row in poi.iterrows()
            ]
        }
    }
    
    resp = requests.post(
        'https://appeears.earthdatacloud.nasa.gov/api/task',
        json=task,
        headers=headers
    )
    task_id = resp.json()
    print(f'Task submitted: {task_id}')

    task_id = resp.json()['task_id']  # extract just the id string

    while True:
        status = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
            headers=headers
        ).json()['status']
        print(f'Status: {status}')
        if status == 'done':
            break
        time.sleep(30)
    
    # Step 5: download results
    files = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
        headers=headers
    ).json()['files']
    
    for f in files:
        if f['file_name'].endswith('.csv'):
            dl = requests.get(
                f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
                headers=headers,
                stream=True
            )
            with open(f['file_name'], 'wb') as out:
                for chunk in dl.iter_content(chunk_size=8192):
                    out.write(chunk)
            print(f'Downloaded: {f["file_name"]}')

In [435]:
land_cover = pd.read_csv('nps-night-lights-MCD12Q1-061-results.csv')

# map numeric codes to names
lc_map = {
    1: 'Evergreen Needleleaf Forest',
    2: 'Evergreen Broadleaf Forest',
    3: 'Deciduous Needleleaf Forest',
    4: 'Deciduous Broadleaf Forest',
    5: 'Mixed Forest',
    6: 'Closed Shrubland',
    7: 'Open Shrubland',
    8: 'Woody Savanna',
    9: 'Savanna',
    10: 'Grassland',
    11: 'Permanent Wetland',
    12: 'Cropland',
    13: 'Urban',
    14: 'Cropland/Natural Mosaic',
    15: 'Snow/Ice',
    16: 'Barren',
    17: 'Water'
}

land_cover = (land_cover
    .rename(columns={
        'ID':                   'park_name',
        'Latitude':                    'latitude',
        'Longitude':                   'longitude',
        'Date':                        'date',
        'MODIS_Tile':                  'modis_tile',
        'MCD12Q1_061_Line_Y_500m':     'line_y',
        'MCD12Q1_061_Sample_X_500m':   'sample_x',
        'MCD12Q1_061_LC_Type1':        'lc_type_code',
        'MCD12Q1_061_QC':              'qc',
        'MCD12Q1_061_QC_bitmask':      'qc_bitmask',
        'MCD12Q1_061_QC_Name':         'qc_name',
        'MCD12Q1_061_QC_Name_Description': 'qc_description',
    }))
    

land_cover['lc_type'] = land_cover['lc_type_code'].map(lc_map)

In [441]:
land_type = land_cover[['park_name', 'lc_type', 'lc_type_code']]
with_land_type = pd.merge(with_TRI, land_type, on = 'park_name')

In [443]:
with_land_type

,county,county_id,state,state_fips,park_code,park_name,fips,population,pop_density,county_x,...,PM2.5 Wtd AM (µg/m3),PM2.5 24-hr (µg/m3),SO2 1-hr (ppb),SO2 Wtd AM (ppb),n_facilities,total_releases,weighted_pollution,max_release,lc_type,lc_type_code
0,Larue County,21123,Kentucky,21,90763,Abraham Lincoln Birthplace National Historical...,21123,14867,21.989644,123,...,NaN,NaN,NaN,NaN,61,717694.692,21084.846952,291540.0,Cropland,12.0
1,Hancock County,23009,Maine,23,49610,"Acadia National Park, US, ME",23009,55478,13.496177,009,...,3.5,12,1,0,1,5405.000,309.399860,5405.0,Evergreen Needleleaf Forest,1.0
2,Hancock County,23009,Maine,23,117673,SERC Education - Schoodic - Acadia National Pa...,23009,55478,13.496177,009,...,3.5,12,1,0,1,5405.000,191.381867,5405.0,Evergreen Needleleaf Forest,1.0
3,Cambria County,42021,Pennsylvania,42,72632,Allegheny Portage Railroad National Historic S...,42021,133472,74.907834,021,...,6.6,15,11,1,125,3231842.377,125045.981823,703103.0,Deciduous Broadleaf Forest,4.0
4,Val Verde County,48465,Texas,48,95085,"Amistad National Recreation Area, US, TX",48465,47586,5.842455,465,...,NaN,NaN,NaN,NaN,6,10257.165,318.452007,6950.0,Grassland,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,Custer County,46033,South Dakota,46,72794,"Wind Cave National Park, US, SD",46033,8318,2.062759,033,...,6.6,22,ND,ND,0,0.000,0.000000,0.0,Grassland,10.0
172,Dare County,37055,North Carolina,37,95344,"Wright Brothers National Memorial, US, NC",37055,36915,37.191761,055,...,NaN,NaN,NaN,NaN,0,0.000,0.000000,0.0,Urban,13.0
173,Park County,56029,Wyoming,56,10211,"Yellowstone National Park, US, WY",56029,29624,1.648339,029,...,3.4,15,ND,ND,0,0.000,0.000000,0.0,Savanna,9.0
174,Mariposa County,06043,California,06,68542,"Yosemite National Park, US, CA",06043,17131,4.565251,043,...,ND,ND,ND,ND,1,0.000,0.000000,0.0,Savanna,9.0


In [447]:
products_to_query = [
    'MCD15A2H.061', 'MCD43A1.061', 'MCD64A1.061', 'MOD09A1.061',
    'MOD10A1.061', 'MOD11A1.061', 'MOD13A1.061', 'MOD14A2.061',
    'MOD16A2.061', 'MOD17A2H.061', 'MOD21A1D.061', 'MOD44B.061',
    'NASADEM_NC.001', 'SPL3SMP_E.006', 'SPL4CMDL.008', 'SPL4SMGP.008',
    'ASTWBD_ATTNC.001', 'VNP16A2.002', 'VNP43IA4.002', 'HLSS30.020',
    'HLSL30_VI.020', 'WaterBalance_Monthly_Historical_GRIDMET.015'
]

# check available layers for each product
for product in products_to_query:
    try:
        resp = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/product/{product}',
            headers=headers
        ).json()
        print(f'\n=== {product} ===')
        print(list(resp.keys())[:5])  # first 5 layers
    except Exception as e:
        print(f'{product}: ERROR - {e}')


=== MCD15A2H.061 ===
['FparExtra_QC', 'FparLai_QC', 'FparStdDev_500m', 'Fpar_500m', 'LaiStdDev_500m']

=== MCD43A1.061 ===
['BRDF_Albedo_Band_Mandatory_Quality_Band1', 'BRDF_Albedo_Band_Mandatory_Quality_Band2', 'BRDF_Albedo_Band_Mandatory_Quality_Band3', 'BRDF_Albedo_Band_Mandatory_Quality_Band4', 'BRDF_Albedo_Band_Mandatory_Quality_Band5']

=== MCD64A1.061 ===
['Burn_Date', 'Burn_Date_Uncertainty', 'First_Day', 'Last_Day', 'QA']

=== MOD09A1.061 ===
['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05']

=== MOD10A1.061 ===
['NDSI_Snow_Cover', 'NDSI_Snow_Cover_Algorithm_Flags_QA', 'NDSI_Snow_Cover_Basic_QA', 'Snow_Albedo_Daily_Tile', 'granule_pnt']

=== MOD11A1.061 ===
['Clear_day_cov', 'Clear_night_cov', 'Day_view_angl', 'Day_view_time', 'Emis_31']

=== MOD13A1.061 ===
['_500m_16_days_EVI', '_500m_16_days_MIR_reflectance', '_500m_16_days_NDVI', '_500m_16_days_NIR_reflectance', '_500m_16_days_VI_Quality']

=== MOD14A2.061 ===
['FireMask', 'QA']

=== MOD16A2

In [449]:
product_layers = {
    'MCD15A2H.061':                          'Fpar_500m',                                        # fraction of photosynthetically active radiation
    'MCD43A1.061':                           'BRDF_Albedo_Band_Mandatory_Quality_Band1',          # surface albedo quality
    'MCD64A1.061':                           'Burn_Date',                                         # fire burn date
    'MOD09A1.061':                           'sur_refl_b02',                                      # NIR surface reflectance
    'MOD10A1.061':                           'NDSI_Snow_Cover',                                   # snow cover
    'MOD11A1.061':                           'LST_Day_1km',                                       # land surface temperature (day)
    'MOD13A1.061':                           '_500m_16_days_NDVI',                                # vegetation greenness
    'MOD14A2.061':                           'FireMask',                                          # fire detection
    'MOD16A2.061':                           'ET_500m',                                           # evapotranspiration
    'MOD17A2H.061':                          'Gpp_500m',                                          # gross primary productivity
    'MOD21A1D.061':                          'LST_1KM',                                           # land surface temperature
    'MOD44B.061':                            'Percent_Tree_Cover',                                # tree cover %
    'NASADEM_NC.001':                        'NASADEM_HGT',                                       # elevation
    'SPL3SMP_E.006':                         'Soil_Moisture_Retrieval_Data_AM_albedo',            # soil moisture
    'SPL4CMDL.008':                          'GPP_gpp_mean',                                      # gross primary production
    'SPL4SMGP.008':                          'Geophysical_Data_depth_to_water_table_from_surface',# water table depth
    'ASTWBD_ATTNC.001':                      'ASTWBD_att',                                        # water body
    'VNP16A2.002':                           'ET_500m',                                           # evapotranspiration (VIIRS)
    'VNP43IA4.002':                          'Nadir_Reflectance_I1',                             # surface reflectance
    'HLSS30.020':                            'B05',                                               # NIR band
    'HLSL30_VI.020':                         'NDVI',                                              # vegetation index
    'WaterBalance_Monthly_Historical_GRIDMET.015': 'AET',                                        # actual evapotranspiration
}

# login once outside the function
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

def NASAapi(product, layer):
    task = {
        'task_type': 'point',
        'task_name': f'nps_{product}',  # unique name per product
        'params': {
            'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
            'layers': [{'product': product, 'layer': layer}],  # use the parameters
            'coordinates': [
                {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
                for _, row in poi.iterrows()
            ]
        }
    }
    
    resp = requests.post(
        'https://appeears.earthdatacloud.nasa.gov/api/task',
        json=task,
        headers=headers
    )
    print(f'Task submitted: {resp.json()}')
    task_id = resp.json()['task_id']
    
    while True:
        status = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
            headers=headers
        ).json()['status']
        print(f'[{product}] Status: {status}')
        if status == 'done':
            break
        time.sleep(30)
    
    files = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
        headers=headers
    ).json()['files']
    
    for f in files:
        if f['file_name'].endswith('.csv'):
            dl = requests.get(
                f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
                headers=headers,
                stream=True
            )
            with open(f['file_name'], 'wb') as out:
                for chunk in dl.iter_content(chunk_size=8192):
                    out.write(chunk)
            print(f'Downloaded: {f["file_name"]}')

# run the loop
for product, layer in product_layers.items():
    print(f'\n--- {product} / {layer} ---')
    NASAapi(product, layer)
    time.sleep(2)


--- MCD15A2H.061 / Fpar_500m ---
Task submitted: {'task_id': 'c4393d0f-4ca9-4705-9d91-97555ec33072', 'status': 'pending'}
[MCD15A2H.061] Status: queued
[MCD15A2H.061] Status: queued
[MCD15A2H.061] Status: queued
[MCD15A2H.061] Status: queued
[MCD15A2H.061] Status: queued
[MCD15A2H.061] Status: queued
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] Status: processing
[MCD15A2H.061] St

KeyboardInterrupt: 

In [451]:
#8-day composite
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'nps_night_lights',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MCD15A2H.061', 'layer': 'Fpar_500m'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')
task_id = resp.json()['task_id']  # extract just the id string

while True:
    status = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
        headers=headers
    ).json()['status']
    print(f'Status: {status}')
    if status == 'done':
        break
    time.sleep(30)

# Step 5: download results
files = requests.get(
    f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
    headers=headers
).json()['files']

for f in files:
    if f['file_name'].endswith('.csv'):
        dl = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
            headers=headers,
            stream=True
        )
        with open(f['file_name'], 'wb') as out:
            for chunk in dl.iter_content(chunk_size=8192):
                out.write(chunk)
        print(f'Downloaded: {f["file_name"]}')

Task submitted: {'task_id': '198b5dff-84d4-426c-8da1-0d936142a677', 'status': 'pending'}
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: pr

In [453]:
# Albedo 
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'nps_night_lights',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MCD43A1.061', 'layer': 'BRDF_Albedo_Band_Mandatory_Quality_Band1'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')
task_id = resp.json()['task_id']  # extract just the id string

while True:
    status = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
        headers=headers
    ).json()['status']
    print(f'Status: {status}')
    if status == 'done':
        break
    time.sleep(30)

# Step 5: download results
files = requests.get(
    f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
    headers=headers
).json()['files']

for f in files:
    if f['file_name'].endswith('.csv'):
        dl = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
            headers=headers,
            stream=True
        )
        with open(f['file_name'], 'wb') as out:
            for chunk in dl.iter_content(chunk_size=8192):
                out.write(chunk)
        print(f'Downloaded: {f["file_name"]}')

Task submitted: {'task_id': '798db19e-444a-4efc-a064-55199afe6d05', 'status': 'pending'}
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: queued
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: pr

KeyboardInterrupt: 

In [454]:
# Burn coverage
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'Burn_coverage',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MCD64A1.061', 'layer': 'Burn_Date'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': '727d2633-0420-4201-be48-13a5d3139559', 'status': 'pending'}


In [456]:
# vegetation greenness
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'vegetation_greenness',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MOD13A1.061', 'layer': '_500m_16_days_NDVI'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')


KeyboardInterrupt: 

In [458]:
# evapotranspiration
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'evapotranspiration',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MOD16A2.061', 'layer': 'ET_500m'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': '82dae215-b091-4a5a-aa78-9c3594a4a016', 'status': 'pending'}


In [460]:
# Gross primary production
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'GPP',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MOD17A2H.061', 'layer': 'Gpp_500m'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': 'a13ccc55-73c3-44a5-9c2e-5d0a328bba29', 'status': 'pending'}


In [ ]:
# Percent Tree cover
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'nps_night_lights',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'MOD44B.061', 'layer': 'Percent_Tree_Cover'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')
task_id = resp.json()['task_id']  # extract just the id string

while True:
    status = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
        headers=headers
    ).json()['status']
    print(f'Status: {status}')
    if status == 'done':
        break
    time.sleep(30)

# Step 5: download results
files = requests.get(
    f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
    headers=headers
).json()['files']

for f in files:
    if f['file_name'].endswith('.csv'):
        dl = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
            headers=headers,
            stream=True
        )
        with open(f['file_name'], 'wb') as out:
            for chunk in dl.iter_content(chunk_size=8192):
                out.write(chunk)
        print(f'Downloaded: {f["file_name"]}')

In [603]:
# Elevation
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'elevation',
    'params': {
        'dates': [{'startDate': '01-01-2000', 'endDate': '01-01-2000'}],
        'layers': [{'product': 'NASADEM_NC.001', 'layer': 'NASADEM_HGT'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': 'e72f082e-ba6c-45fc-b6f6-6c5892ded640', 'status': 'pending'}


In [ ]:
# water table depth
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'nps_night_lights',
    'params': {
        'dates': [{'startDate': '01-01-2021', 'endDate': '12-31-2021'}],
        'layers': [{'product': 'SPL4SMGP.008', 'layer': 'Geophysical_Data_depth_to_water_table_from_surface'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')
task_id = resp.json()['task_id']  # extract just the id string

while True:
    status = requests.get(
        f'https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}',
        headers=headers
    ).json()['status']
    print(f'Status: {status}')
    if status == 'done':
        break
    time.sleep(30)

# Step 5: download results
files = requests.get(
    f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}',
    headers=headers
).json()['files']

for f in files:
    if f['file_name'].endswith('.csv'):
        dl = requests.get(
            f'https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}/{f["file_id"]}',
            headers=headers,
            stream=True
        )
        with open(f['file_name'], 'wb') as out:
            for chunk in dl.iter_content(chunk_size=8192):
                out.write(chunk)
        print(f'Downloaded: {f["file_name"]}')

In [464]:
# snow cover
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'snow_cover_jan',
    'params': {
        'dates': [{'startDate': '01-15-2021', 'endDate': '01-16-2021'}],
        'layers': [{'product': 'MOD10A1.061', 'layer': 'NDSI_Snow_Cover'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': '0a58c670-5004-406a-bd1a-58b70b3a705b', 'status': 'pending'}


In [466]:
# soil moisture
EARTHDATA_USER = 'isabellawoods'
EARTHDATA_PASS = 'Atlanta!1101'

# get token
token_resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/login',
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
token = token_resp.json()['token']
headers = {'Authorization': f'Bearer {token}'}

# Step 2: submit a point sample task for all parks
task = {
    'task_type': 'point',
    'task_name': 'soil_moisture',
    'params': {
        'dates': [{'startDate': '04-15-2021', 'endDate': '04-16-2021'}],
        'layers': [{'product': 'SPL3SMP_E.006', 'layer': 'Soil_Moisture_Retrieval_Data_AM_albedo'}],
        'coordinates': [
            {'id': str(row['display_name']), 'latitude': str(row['latitude']), 'longitude': str(row['longitude'])}
            for _, row in poi.iterrows()
        ]
    }
}

resp = requests.post(
    'https://appeears.earthdatacloud.nasa.gov/api/task',
    json=task,
    headers=headers
)
task_id = resp.json()
print(f'Task submitted: {task_id}')

Task submitted: {'task_id': '65fbda11-fa00-47c1-bdda-8c76403e4deb', 'status': 'pending'}


In [599]:
name_mapping = {
    'Abraham Lincoln Birthplace National Historical Park, US, KY': 'Abraham Lincoln Birthplace NHP',
    'Acadia National Park, US, ME': 'Acadia NP',
    'SERC Education - Schoodic - Acadia National Park, US, ME': 'Acadia NP',
    'Allegheny Portage Railroad National Historic Site, US, PA': 'Allegheny Portage Railroad NHS',
    'Amistad National Recreation Area, US, TX': 'Amistad NRA',
    'Apostle Islands National Lakeshore, US, WI': 'Apostle Islands NL',
    'Appomattox Court House National Historical Park, US, VA': 'Appomattox Court House NHP',
    'Arches National Park, US, UT': 'Arches NP',
    'Arkansas Post National Memorial, US, AR': 'Arkansas Post NMEM',
    'Badlands National Park, US, SD': 'Badlands NP',
    'Big Bend National Park, US, TX': 'Big Bend NP',
    'Big Cypress National Preserve, US, FL': 'Big Cypress NPRES',
    'Big Hole National Battlefield, US, MT': 'Big Hole NB',
    'Big Thicket National Preserve, US, TX': 'Big Thicket NPRES',
    'Bighorn Canyon National Recreation Area, US, MT': 'Bighorn Canyon NRA',
    'Biscayne National Park, US, FL': 'Biscayne NP',
    'Black Canyon of the Gunnison National Park, US, CO': 'Black Canyon of the Gunnison NP',
    'Booker T. Washington National Monument, US, VA': 'Booker T. Washington NM',
    'Bryce Canyon National Park, US, UT': 'Bryce Canyon NP',
    'Buffalo National River, US, AR': 'Buffalo NR',
    'Cabrillo National Monument, US, CA': 'Cabrillo NM',
    'Canaveral National Seashore, US, FL': 'Canaveral NS',
    'Cane River Creole National Historical Park, US, LA': 'Cane River Creole NHP',
    'Canyon de Chelly National Monument, US, AZ': 'Canyon de Chelly NM',
    'Canyonlands National Park, US, UT': 'Canyonlands NP',
    'Cape Cod National Seashore, US, MA': 'Cape Cod NS',
    'Cape Hatteras National Seashore, US, NC': 'Cape Hatteras NS',
    'Cape Lookout National Seashore, US, NC': 'Cape Lookout NS',
    'Capitol Reef National Park, US, UT': 'Capitol Reef NP',
    'Capulin Volcano National Monument, US, NM': 'Capulin Volcano NM',
    'Carlsbad Caverns National Park, US, NM': 'Carlsbad Caverns NP',
    'Catoctin Mountain Park, US, MD': 'Catoctin Mountain Park',
    'Cedar Breaks National Monument, US, UT': 'Cedar Breaks NM',
    'Chaco Culture National Historical Park, US, NM': 'Chaco Culture NHP',
    'Charles Pinckney National Historic Site, US, SC': 'Charles Pinckney NHS',
    'Chattahoochee River National Recreation Area, US, GA': 'Chattahoochee River NRA',
    'Chickamauga and Chattanooga National Military Park, US, GA': 'Chickamauga & Chattanooga NMP',
    'Chickasaw National Recreation Area, US, OK': 'Chickasaw NRA',
    'City of Rocks National Reserve, US, ID': 'City of Rocks NRES',
    'Colonial National Historical Park, US, VA': 'Colonial NHP',
    'Colorado National Monument, US, CO': 'Colorado NM',
    'Congaree National Park, US, SC': 'Congaree NP',
    'Coronado National Memorial, US, AZ': 'Coronado NMEM',
    'Cowpens National Battlefield, US, SC': 'Cowpens NB',
    'Crater Lake National Park, US, OR': 'Crater Lake NP',
    'Cumberland Gap National Historical Park, US, KY': 'Cumberland Gap NHP',
    'Curecanti National Recreation Area, US, CO': 'Curecanti NRA',
    'Cuyahoga Valley National Park, US, OH': 'Cuyahoga Valley NP',
    'De Soto National Memorial, US, FL': 'De Soto NMEM',
    'Death Valley National Park, US, CA': 'Death Valley NP',
    'Devils Postpile National Monument, US, CA': 'Devils Postpile NM',
    'Devils Tower National Monument, US, WY': 'Devils Tower NM',
    'Dinosaur National Monument, US, CO': 'Dinosaur NM',
    'Eisenhower National Historic Site, US, PA': 'Eisenhower NHS',
    'El Malpais National Monument, US, NM': 'El Malpais NM',
    'El Morro National Monument, US, NM': 'El Morro NM',
    'Everglades National Park, US, FL': 'Everglades NP',
    'Fire Island National Seashore, US, NY': 'Fire Island NS',
    'Florissant Fossil Beds National Monument, US, CO': 'Florissant Fossil Beds NM',
    'Fort Caroline National Memorial, US, FL': 'Fort Caroline NMEM',
    'Fort Frederica National Monument, US, GA': 'Fort Frederica NM',
    'Fort Laramie National Historic Site, US, WY': 'Fort Laramie NHS',
    'Fort Matanzas National Monument, US, FL': 'Fort Matanzas NM',
    'Fort Necessity National Battlefield, US, PA': 'Fort Necessity NB',
    'Fort Point National Historic Site, US, CA': 'Fort Point NHS',
    'Fort Pulaski National Monument, US, GA': 'Fort Pulaski NM',
    'Fort Raleigh National Historic Site, US, NC': 'Fort Raleigh NHS',
    'Fort Vancouver National Historic Site, US, WA': 'Fort Vancouver NHS',
    'Fossil Butte National Monument, US, WY': 'Fossil Butte NM',
    'Fredericksburg and Spotsylvania National Military Park, US, VA': 'Fredericksburg & Spotsylvania NMP',
    'Gauley River National Recreation Area, US, WV': 'Gauley River NRA',
    'George Washington Birthplace National Monument, US, VA': 'George Washington Birthplace NM',
    'George Washington Carver National Monument, US, MO': 'George Washington Carver NM',
    'Gettysburg National Military Park, US, PA': 'Gettysburg NMP',
    'Glacier National Park, US, MT': 'Glacier NP',
    'Golden Gate National Recreation Area, US, CA': 'Golden Gate NRA',
    'Grand Canyon National Park, US, AZ': 'Grand Canyon NP',
    'Grand Portage National Monument, US, MN': 'Grand Portage NM',
    'Grand Teton National Park, US, WY': 'Grand Teton NP',
    'Great Basin National Park, US, NV': 'Great Basin NP',
    'Guadalupe Mountains National Park, US, TX': 'Guadalupe Mountains NP',
    'Guilford Courthouse National Military Park, US, NC': 'Guilford Courthouse NMP',
    'Gulf Islands National Seashore, US, FL': 'Gulf Islands NS',
    'Haleakala National Park, US, HI': 'Haleakala NP',
    'Hawaii Volcanoes National Park, US, HI': 'Hawaii Volcanoes NP',
    'Homestead National Historical Park, US, NE': 'Homestead NHP',
    'Hopewell Culture National Historical Park, US, OH': 'Hopewell Culture NHP',
    'Hopewell Furnace National Historic Site, US, PA': 'Hopewell Furnace NHS',
    'Horseshoe Bend National Military Park, US, AL': 'Horseshoe Bend NMP',
    'Hot Springs National Park, US, AR': 'Hot Springs NP',
    'Hubbell Trading Post National Historic Site, US, AZ': 'Hubbell Trading Post NHS',
    'Jean Lafitte National Historical Park and Preserve, US, LA': 'Jean Lafitte NHP & PRES',
    'Jewel Cave National Monument, US, SD': 'Jewel Cave NM',
    'John Day Fossil Beds National Monument, US, OR': 'John Day Fossil Beds NM',
    'Johnstown Flood National Memorial, US, PA': 'Johnstown Flood NMEM',
    'Joshua Tree National Park, US, CA': 'Joshua Tree NP',
    'Kenai Fjords National Park, US, AK': 'Kenai Fjords NP',
    'Kennesaw Mountain National Battlefield Park, US, GA': 'Kennesaw Mountain NBP',
    'Lake Meredith National Recreation Area, US, TX': 'Lake Meredith NRA',
    'Lake Roosevelt National Recreation Area, US, WA': 'Lake Roosevelt NRA',
    'Lassen Volcanic National Park, US, CA': 'Lassen Volcanic NP',
    'Lava Beds National Monument, US, CA': 'Lava Beds NM',
    'Lewis and Clark National Historical Park, US, OR': 'Lewis & Clark NHP',
    'Lincoln Boyhood National Memorial, US, IN': 'Lincoln Boyhood NMEM',
    'Little Bighorn Battlefield National Monument, US, MT': 'Little Bighorn Battlefield NM',
    'Little River Canyon National Preserve, US, AL': 'Little River Canyon NPRES',
    'Lyndon B. Johnson National Historical Park, US, TX': 'Lyndon B. Johnson NHP',
    'Mammoth Cave National Park, US, KY': 'Mammoth Cave NP',
    'Manassas National Battlefield Park, US, VA': 'Manassas NBP',
    'Manzanar National Historic Site, US, CA': 'Manzanar NHS',
    'Martin Van Buren National Historic Site, US, NY': 'Martin Van Buren NHS',
    'Mesa Verde National Park, US, CO': 'Mesa Verde NP',
    'Minute Man National Historical Park, US, MA': 'Minute Man NHP',
    'Mojave National Preserve, US, CA': 'Mojave NPRES',
    'Monocacy National Battlefield, US, MD': 'Monocacy NB',
    'Montezuma Castle National Monument, US, AZ': 'Montezuma Castle NM',
    'Moores Creek National Battlefield, US, NC': 'Moores Creek NB',
    'Morristown National Historical Park, US, NJ': 'Morristown NHP',
    'Mount Rainier National Park, US, WA': 'Mount Rainier NP',
    'Mount Rushmore National Memorial, US, SD': 'Mount Rushmore NMEM',
    'Natural Bridges National Monument, US, UT': 'Natural Bridges NM',
    'Navajo National Monument, US, AZ': 'Navajo NM',
    'New River Gorge National Park and Preserve, US, WV': 'New River Gorge NP & PRES',
    'Nez Perce National Historical Park, US, ID': 'Nez Perce NHP',
    'North Cascades National Park, US, WA': 'North Cascades NP',
    'Organ Pipe Cactus National Monument, US, AZ': 'Organ Pipe Cactus NM',
    'Padre Island National Seashore, US, TX': 'Padre Island NS',
    'Palo Alto Battlefield National Historical Park, US, TX': 'Palo Alto Battlefield NHP',
    'Pea Ridge National Military Park, US, AR': 'Pea Ridge NMP',
    'Petersburg National Battlefield, US, VA': 'Petersburg NB',
    'Petrified Forest National Park, US, AZ': 'Petrified Forest NP',
    'Petroglyph National Monument, US, NM': 'Petroglyph NM',
    'Pictured Rocks National Lakeshore, US, MI': 'Pictured Rocks NL',
    'Pinnacles National Park, US, CA': 'Pinnacles NP',
    'Pipe Spring National Monument, US, AZ': 'Pipe Spring NM',
    'Piscataway Park, US, MD': 'Piscataway Park',
    'Point Reyes National Seashore, US, CA': 'Point Reyes NS',
    'Prince William Forest Park, US, VA': 'Prince William Forest Park',
    "Pu'ukohola Heiau National Historic Site, US, HI": "Pu'ukohola Heiau NHS",
    'Redwood National Park, US, CA': 'Redwood NP',
    'Richmond National Battlefield Park, US, VA': 'Richmond NBP',
    'Rocky Mountain National Park, US, CO': 'Rocky Mountain NP',
    'Russell Cave National Monument, US, AL': 'Russell Cave NM',
    'Saguaro National Park, US, AZ': 'Saguaro NP',
    'San Antonio Missions National Historical Park, US, TX': 'San Antonio Missions NHP',
    'San Juan Island National Historical Park, US, WA': 'San Juan Island NHP',
    'Santa Monica Mountains National Recreation Area, US, CA': 'Santa Monica Mountains NRA',
    'Saratoga National Historical Park, US, NY': 'Saratoga NHP',
    'Scotts Bluff National Monument, US, NE': 'Scotts Bluff NM',
    'Shenandoah National Park, US, VA': 'Shenandoah NP',
    'Sleeping Bear Dunes National Lakeshore, US, MI': 'Sleeping Bear Dunes NL',
    'Sunset Crater Volcano National Monument, US, AZ': 'Sunset Crater Volcano NM',
    'Theodore Roosevelt National Park, US, ND': 'Theodore Roosevelt NP',
    'Timucuan Ecological and Historic Preserve, US, FL': 'Timucuan EHP',
    'Tonto National Monument, US, AZ': 'Tonto NM',
    'Tuzigoot National Monument, US, AZ': 'Tuzigoot NM',
    'Upper Delaware Scenic and Recreational River, US, NY': 'Upper Delaware S&RR',
    'Valley Forge National Historical Park, US, PA': 'Valley Forge NHP',
    'Vanderbilt Mansion National Historic Site, US, NY': 'Vanderbilt Mansion NHS',
    'Walnut Canyon National Monument, US, AZ': 'Walnut Canyon NM',
    'Washita Battlefield National Historic Site, US, OK': 'Washita Battlefield NHS',
    'White Sands National Park, US, NM': 'White Sands NP',
    'Whitman Mission National Historic Site, US, WA': 'Whitman Mission NHS',
    "Wilson's Creek National Battlefield, US, MO": "Wilson's Creek NB",
    'Wind Cave National Park, US, SD': 'Wind Cave NP',
    'Wright Brothers National Memorial, US, NC': 'Wright Brothers NMEM',
    'Yellowstone National Park, US, WY': 'Yellowstone NP',
    'Yosemite National Park, US, CA': 'Yosemite NP',
    'Zion National Park, US, UT': 'Zion NP',
}

# apply mapping and merge
with_land_type['visit_name'] = with_land_type['park_name'].map(name_mapping)

merged = with_land_type.merge(
    visit_traffic_ft,
    left_on='visit_name',
    right_on='ParkName',
    how='left'
)

# check how many matched
print(f'Total rows: {len(merged)}')
print(f'Matched: {merged["ParkName"].notna().sum()}')
print(f'Unmatched: {merged["ParkName"].isna().sum()}')
print(merged[merged["ParkName"].isna()][['park_name', 'visit_name']])

Total rows: 176
Matched: 176
Unmatched: 0
Empty DataFrame
Columns: [park_name, visit_name]
Index: []


# Data from NASA

### Burn Coverage

In [616]:
burn = pd.read_csv('DATA/Burn-coverage-MCD64A1-061-results.csv')
burn = burn.rename(columns={
    'ID':                        'park_name',
    'MCD64A1_061_Burn_Date':     'burn_date'})

burn = burn[['park_name', 'Date', 'burn_date']]

In [624]:
burn['burned'] = (burn['burn_date'] > 0).astype(int)

burn_features = burn.groupby('park_name').agg(
    n_burn_observations = ('burned', 'sum'),       # how many months had fire
    pct_burned =          ('burned', 'mean'),       # proportion of observations with fire
    earliest_burn =       ('burn_date', lambda x: x[x > 0].min() if (x > 0).any() else 0),  # earliest burn day of year
    latest_burn =         ('burn_date', lambda x: x[x > 0].max() if (x > 0).any() else 0),  # latest burn day of year
).reset_index()

with_burns = pd.merge(merged, burn_features, on = "park_name", how = "left")

### Evapotranspiration

In [679]:
et = pd.read_csv('DATA/evapotranspiration-MOD16A2-061-results.csv')
et = et.rename(columns={
    'ID':                    'park_name',
    'Date':                  'date',
    'MOD16A2_061_ET_500m':   'ET',
    'MOD16A2_061_ET_QC_500m_MODLAND': 'quality',
})

print(et['quality'].unique())  # see what values exist

# filter using the description column instead
et_raw = pd.read_csv('DATA/evapotranspiration-MOD16A2-061-results.csv')  # reload raw

et = et_raw.rename(columns={
    'ID':                    'park_name',
    'Date':                  'date',
    'MOD16A2_061_ET_500m':   'ET',
    'MOD16A2_061_ET_QC_500m_MODLAND':             'quality',
    'MOD16A2_061_ET_QC_500m_MODLAND_Description': 'quality_desc',
})

et = et[['park_name', 'date', 'ET', 'quality', 'quality_desc']]

# check what descriptions exist
print(et['quality_desc'].unique())

# filter to good quality using description
et = et[et['quality_desc'].str.contains('good|produced', case=False, na=False)]

# convert ET
et['ET'] = pd.to_numeric(et['ET'], errors='coerce') * 0.1

et_features = et.groupby('park_name').agg(
    avg_ET =   ('ET', 'mean'),
    max_ET =   ('ET', 'max'),
    min_ET =   ('ET', 'min'),
    ET_range = ('ET', lambda x: x.max() - x.min()),
).reset_index()

with_et = pd.merge(with_burns, et_features, on = "park_name", how = "left")

['0b0' '0b1']
['Good quality (main algorithm with or without saturation)'
 'Other Quality (back-up algorithm or fill values)']


In [720]:
gpp = pd.read_csv('DATA/GPP-MOD17A2H-061-results.csv')

gpp = gpp.rename(columns={
    'ID':                    'park_name',
    'Date':                  'date',
    'MOD17A2H_061_Gpp_500m':   'GPP'
})
gpp = gpp[['park_name', 'date', 'GPP']]
gpp = gpp.groupby('park_name').agg(
    avg_GPP =   ('GPP', 'mean'),
    max_GPP =   ('GPP', 'max'),
    min_GPP =   ('GPP', 'min'),
    GPP_range = ('GPP', lambda x: x.max() - x.min()),
).reset_index()

with_gpp = pd.merge(with_et, gpp, on = "park_name", how = "left")

In [722]:
snow = pd.read_csv('DATA/snow-cover-jan-MOD10A1-061-results.csv')

snow = snow.rename(columns={
    'ID':                    'park_name',
    'Date':                  'date',
    'MOD10A1_061_NDSI_Snow_Cover': 'SNOW'
})
snow = snow[['park_name', 'date', 'SNOW']]

snow = snow.groupby('park_name').agg(
    avg_SNOW =   ('SNOW', 'mean'),
    max_SNOW =   ('SNOW', 'max'),
    min_SNOW =   ('SNOW', 'min'),
    SNOW_range = ('SNOW', lambda x: x.max() - x.min()),
).reset_index()

with_snow = pd.merge(with_gpp, snow, on = "park_name", how = "left")
with_snow.head()

,county,county_id,state,state_fips,park_code,park_name,fips,population,pop_density,county_x,...,min_ET,ET_range,avg_GPP,max_GPP,min_GPP,GPP_range,avg_SNOW,max_SNOW,min_SNOW,SNOW_range
0,Larue County,21123,Kentucky,21,90763,Abraham Lincoln Birthplace National Historical...,21123,14867,21.989644,123,...,0.37,2.80,0.025487,0.0611,0.0007,0.0604,125.0,250.0,0.0,250.0
1,Hancock County,23009,Maine,23,49610,"Acadia National Park, US, ME",23009,55478,13.496177,009,...,0.20,3.53,0.035687,0.0954,0.0007,0.0947,250.0,250.0,250.0,0.0
2,Hancock County,23009,Maine,23,117673,SERC Education - Schoodic - Acadia National Pa...,23009,55478,13.496177,009,...,0.26,3.57,0.031272,0.0892,0.0003,0.0889,250.0,250.0,250.0,0.0
3,Cambria County,42021,Pennsylvania,42,72632,Allegheny Portage Railroad National Historic S...,42021,133472,74.907834,021,...,0.28,3.18,0.036670,0.0969,0.0000,0.0969,250.0,250.0,250.0,0.0
4,Val Verde County,48465,Texas,48,95085,"Amistad National Recreation Area, US, TX",48465,47586,5.842455,465,...,0.21,0.95,0.012113,0.0228,0.0034,0.0194,0.0,0.0,0.0,0.0


In [726]:
soil = pd.read_csv('DATA/soil-moisture-SPL3SMP-E-006-results.csv')

soil = soil.rename(columns={
    'ID':   'park_name',
    'Date': 'date',
    'SPL3SMP_E_006_Soil_Moisture_Retrieval_Data_AM_albedo': 'soil_moisture',
    'SPL3SMP_E_006_Soil_Moisture_Retrieval_Data_AM_retrieval_qual_flag_Recommended_Quality_Description': 'quality_desc',
})

soil = soil[['park_name', 'date', 'soil_moisture', 'quality_desc']]

# filter to recommended quality only
soil = soil[soil['quality_desc'].str.contains('recommended', case=False, na=False)]

soil['soil_moisture'] = pd.to_numeric(soil['soil_moisture'], errors='coerce')

soil_features = soil_features = soil.groupby('park_name').agg(
    avg_soil_moisture =   ('soil_moisture', 'mean'),
    max_soil_moisture =   ('soil_moisture', 'max'),
    min_soil_moisture =   ('soil_moisture', 'min'),
    soil_moisture_range = ('soil_moisture', lambda x: x.max() - x.min()),
).reset_index()

with_soil = pd.merge(with_snow, soil_features, on='park_name', how='left')

In [734]:
ps = pd.read_csv('DATA/nps-night-lights-MCD15A2H-061-results.csv')
ps = ps.rename(columns={
    'ID':                    'park_name',
    'Date':                  'date',
    'MCD15A2H_061_Fpar_500m': 'FPAR',
    'MCD15A2H_061_FparLai_QC_MODLAND_Description': 'quality_desc',
})

ps = ps[['park_name', 'date', 'FPAR', 'quality_desc']]

# filter to good quality
ps = ps[ps['quality_desc'].str.contains('good|produced', case=False, na=False)]

ps['FPAR'] = pd.to_numeric(ps['FPAR'], errors='coerce') * 0.01  # FPAR is scaled by 0.01

ps_features = ps.groupby('park_name').agg(
    avg_FPAR =   ('FPAR', 'mean'),
    max_FPAR =   ('FPAR', 'max'),
    min_FPAR =   ('FPAR', 'min'),
    FPAR_range = ('FPAR', lambda x: x.max() - x.min()),
).reset_index()

with_ps = pd.merge(with_soil, ps_features, on='park_name', how='left')
with_ps

,county,county_id,state,state_fips,park_code,park_name,fips,population,pop_density,county_x,...,min_SNOW,SNOW_range,avg_soil_moisture,max_soil_moisture,min_soil_moisture,soil_moisture_range,avg_FPAR,max_FPAR,min_FPAR,FPAR_range
0,Larue County,21123,Kentucky,21,90763,Abraham Lincoln Birthplace National Historical...,21123,14867,21.989644,123,...,0.0,250.0,-4999.459752,0.080496,-9999.000000,9999.080496,0.004959,0.0079,0.0030,0.0049
1,Hancock County,23009,Maine,23,49610,"Acadia National Park, US, ME",23009,55478,13.496177,009,...,250.0,0.0,-4999.465000,0.070000,-9999.000000,9999.070000,0.007585,0.0094,0.0040,0.0054
2,Hancock County,23009,Maine,23,117673,SERC Education - Schoodic - Acadia National Pa...,23009,55478,13.496177,009,...,250.0,0.0,-4999.465000,0.070000,-9999.000000,9999.070000,0.006764,0.0095,0.0022,0.0073
3,Cambria County,42021,Pennsylvania,42,72632,Allegheny Portage Railroad National Historic S...,42021,133472,74.907834,021,...,250.0,0.0,-4999.458719,0.082562,-9999.000000,9999.082562,0.005944,0.0093,0.0013,0.0080
4,Val Verde County,48465,Texas,48,95085,"Amistad National Recreation Area, US, TX",48465,47586,5.842455,465,...,0.0,0.0,-9999.000000,-9999.000000,-9999.000000,0.000000,0.003023,0.0043,0.0016,0.0027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,Custer County,46033,South Dakota,46,72794,"Wind Cave National Park, US, SD",46033,8318,2.062759,033,...,0.0,0.0,-4999.465000,0.070000,-9999.000000,9999.070000,0.003785,0.0058,0.0007,0.0051
172,Dare County,37055,North Carolina,37,95344,"Wright Brothers National Memorial, US, NC",37055,36915,37.191761,055,...,0.0,0.0,-9999.000000,-9999.000000,-9999.000000,0.000000,NaN,NaN,NaN,NaN
173,Park County,56029,Wyoming,56,10211,"Yellowstone National Park, US, WY",56029,29624,1.648339,029,...,250.0,0.0,0.070000,0.070000,0.070000,0.000000,0.004388,0.0065,0.0010,0.0055
174,Mariposa County,06043,California,06,68542,"Yosemite National Park, US, CA",06043,17131,4.565251,043,...,30.0,39.0,-4999.464835,0.070331,-9999.000000,9999.070331,0.002662,0.0040,0.0006,0.0034


### MERGE in RESPONSE variables

In [771]:
df = pd.merge(sdi, with_ps, on='park_name', how='left')

In [754]:
data.to_csv('full_data.csv', index=False)

park_name               object
SDI                    float64
n_observations           int64
n_species                int64
county                  object
                        ...   
soil_moisture_range    float64
avg_FPAR               float64
max_FPAR               float64
min_FPAR               float64
FPAR_range             float64
Length: 93, dtype: object

In [773]:
# columns to drop (identifiers, duplicates, not useful as features)
drop_cols = [
    'park_name', 'county', 'county_id', 'state', 'state_fips', 'fips',
    'county_x', 'State', 'County', 'visit_name', 'Code', 'UnitCode', 'ParkName'
]

# columns to convert to numeric (air quality + population)
convert_cols = [
    'population', 'Population (2020 Census)',
    'CO          8-hr (ppm)', 'Pb           3-mo (µg/m3)',
    'NO2         AM (ppb)', 'NO2          1-hr (ppb)',
    'O3            8-hr (ppm)', 'PM10        24-hr (µg/m3) ',
    'PM2.5     Wtd AM (µg/m3) ', 'PM2.5     24-hr (µg/m3) ',
    'SO2         1-hr (ppb)', 'SO2    Wtd AM (ppb)'
]

# clean and convert
for col in convert_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '').str.strip().replace({'ND': None, 'IN': None, 'nan': None}),
        errors='coerce'
    )

# rename messy air quality columns
df = df.rename(columns={
    'Population (2020 Census)':      'population_2020',
    'CO          8-hr (ppm)':        'CO_8hr',
    'Pb           3-mo (µg/m3)':     'Pb_3mo',
    'NO2         AM (ppb)':          'NO2_AM',
    'NO2          1-hr (ppb)':       'NO2_1hr',
    'O3            8-hr (ppm)':      'O3_8hr',
    'PM10        24-hr (µg/m3) ':    'PM10_24hr',
    'PM2.5     Wtd AM (µg/m3) ':     'PM25_wtd',
    'PM2.5     24-hr (µg/m3) ':      'PM25_24hr',
    'SO2         1-hr (ppb)':        'SO2_1hr',
    'SO2    Wtd AM (ppb)':           'SO2_wtd',
})

# one-hot encode land cover type since it's categorical
df = pd.get_dummies(df, columns=['lc_type'], prefix='lc')

# drop identifier columns
df = df.drop(columns=drop_cols, errors='ignore')

# verify no more object columns
print(df.dtypes[df.dtypes == 'object'])
print(df.shape)

Series([], dtype: object)
(152, 95)


In [ ]:
df.to_csv('full_data_clean.csv', index=False)